In [1]:
from numpy import mean
from numpy import std
from numpy import absolute
import pandas as pd
# use automatically configured elastic net algorithm
from numpy import arange
from sklearn.model_selection import cross_val_score
#from sklearn.model_selection import RepeatedKFold
from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import ElasticNet
#from sklearn.linear_model import ElasticNet
from sklearn.model_selection import RepeatedKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import pearsonr
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
import matplotlib.pyplot as plt
import statsmodels.api as sm
lowess = sm.nonparametric.lowess
from sklearn.model_selection import GridSearchCV
import seaborn as sns
from scipy import stats


In [2]:
import sklearn
print(sklearn.__version__)


1.6.0


In [3]:
df_norm = pd.read_csv("/home/lajoyce/Documents/african_killifish_atlas/final_analysis/AtlasFiles_forLajoyce_240507/CountsNormDESeq2_AllTissue_240506.csv")

In [4]:
df_norm = df_norm.T

In [5]:
df_metadata = pd.read_csv("/home/lajoyce/Documents/african_killifish_atlas/final_analysis/AtlasFiles_forLajoyce_240507/ExperimentDesign_allbatches_combined_v7.csv")

df_metadata.head(3)

,Unnamed: 0,animalID,sex,cohort,age_days,harvest_date,hatch_date,tissue,tissue_grind_date,RNA_extract_date,RNA_batch,RNA_extractor,RNAID,cDNA_batch,plate_well,sampleNames,lib,censored,censor_code,notes
0,A1_1,J9,F,2,155,8/16/22,3/14/22,Gut,NaN,3/8/23,Gut_1,EC,RNA352,1,A1,A1,A1_1,NaN,NaN,NaN
1,A1_2,A01,M,2,78,5/31/22,3/14/22,Bone,4/19/23,4/22/23,Bone_1,EC,RNA572,2,A1,A1,A1_2,NaN,NaN,NaN
2,A10_1,P_1B_10,M,1B,133,1/31/22,9/20/21,Kidney,NaN,3/4/23,Kidney_1,JC,R258,1,A10,A10,A10_1,NaN,NaN,NaN


In [6]:
df_metadata.rename(columns={"Unnamed: 0": "sample"}, inplace=True)

df_metadata.head(3)

,sample,animalID,sex,cohort,age_days,harvest_date,hatch_date,tissue,tissue_grind_date,RNA_extract_date,RNA_batch,RNA_extractor,RNAID,cDNA_batch,plate_well,sampleNames,lib,censored,censor_code,notes
0,A1_1,J9,F,2,155,8/16/22,3/14/22,Gut,NaN,3/8/23,Gut_1,EC,RNA352,1,A1,A1,A1_1,NaN,NaN,NaN
1,A1_2,A01,M,2,78,5/31/22,3/14/22,Bone,4/19/23,4/22/23,Bone_1,EC,RNA572,2,A1,A1,A1_2,NaN,NaN,NaN
2,A10_1,P_1B_10,M,1B,133,1/31/22,9/20/21,Kidney,NaN,3/4/23,Kidney_1,JC,R258,1,A10,A10,A10_1,NaN,NaN,NaN


In [7]:
df_metadata.set_index("sample", inplace=True)

In [8]:
# Merge based on index
combined_df = pd.concat([df_norm, df_metadata[['sex', 'tissue', 'age_days']]], axis=1)

combined_df.head(10)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A1_1,23876.707871,687.004459,2181.018965,576.907591,221.294706,884.077854,1368.504075,1348.686639,619.845370,636.359900,...,243.314079,1718.612117,47.341653,568.099841,3323.824460,1566.678439,358.915791,F,Gut,155
A1_2,36.845369,407.384647,349.683408,374.015256,493.588906,469.257059,963.541161,1619.805848,453.962755,449.791581,...,107.060129,777.923924,19.465478,672.254187,2691.797529,191.873997,253.746410,M,Bone,78
A10_1,4821.965545,907.596706,218.510659,554.051370,191.503723,684.175695,1125.288972,662.897504,586.787050,637.527352,...,112.938093,1122.015404,311.807344,542.593883,2612.307200,408.377598,391.191366,M,Kidney,133
A10_2,4.382338,451.380822,1198.256441,303.633424,289.860362,445.120339,1093.080327,468.284126,709.312721,492.073962,...,174.667475,1304.058604,28.172173,1264.617561,1144.416288,157.764171,238.524401,M,SpinalCord,75
A11_1,7011.583272,766.674115,222.103168,627.278822,272.285474,675.602523,1267.567871,975.767056,739.724358,713.703904,...,109.657631,1059.404232,398.670540,521.338398,1948.746206,360.569160,446.064940,M,Kidney,47
A11_2,8.609836,486.229181,1020.945335,358.441083,187.603803,594.985009,1253.410917,559.639365,897.235581,560.545663,...,111.474724,1103.871654,52.565317,1282.865621,983.787094,216.605357,251.951001,F,SpinalCord,52
A12_1,7322.278018,732.665214,225.267215,592.693352,260.260180,651.743981,1162.787081,875.553156,586.132171,640.808680,...,111.540077,1065.827406,275.569603,626.957297,1945.025662,390.025761,450.534430,M,Kidney,161
A12_2,6.381242,390.531984,1030.570512,342.672672,241.849055,465.830634,1290.287044,752.986504,714.699055,485.612483,...,134.644197,1193.930296,27.439339,1560.851686,1036.951754,135.282321,209.304723,M,SpinalCord,161
A13_1,2.350903,453.724234,134.001458,712.323538,481.935067,1024.993606,886.290343,2868.101375,759.341594,702.919927,...,96.387013,883.939440,14.105417,1288.294716,1539.841312,298.564651,347.933609,M,Muscle,134
A13_2,26.477799,505.640544,194.739939,536.388955,640.591905,766.147919,1105.234567,997.615127,667.923826,527.847730,...,213.530635,1276.059075,9.395348,825.082374,3853.800902,227.196596,538.951323,M,Heart,162


# Gut

In [9]:
gut_df = combined_df.loc[combined_df.iloc[:,-2] == "Gut"]

gut_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A1_1,23876.707871,687.004459,2181.018965,576.907591,221.294706,884.077854,1368.504075,1348.686639,619.845370,636.359900,...,243.314079,1718.612117,47.341653,568.099841,3323.824460,1566.678439,358.915791,F,Gut,155
A2_1,23741.305346,614.790132,4013.251397,576.451327,286.171799,729.806549,1381.566243,1759.477327,503.881445,453.219451,...,214.971160,1263.811341,50.661993,412.142160,4515.763598,2272.943473,276.587097,M,Gut,133
A3_1,25474.794219,394.841817,2380.296399,618.502214,223.660397,730.957162,1441.922333,1626.848248,689.723681,502.298768,...,252.398883,1593.111763,31.237486,541.033250,3708.514285,2226.607970,314.873854,F,Gut,134
A4_1,21094.343422,503.698033,2804.802732,581.903780,319.450595,775.429867,1249.966435,1181.039336,534.185019,514.302202,...,172.317748,1569.417030,86.158874,393.679779,3952.704039,1851.753033,279.684961,F,Gut,75
B1_1,26292.108219,472.266501,1996.967953,731.856697,308.067883,889.800129,1341.737278,1681.081088,476.957890,440.990574,...,148.560654,1469.968579,10.946575,688.070399,3907.927106,1620.093030,320.578254,M,Gut,134


In [10]:
gut_df_male = gut_df.loc[gut_df.iloc[:,-3] == "M"]

gut_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A2_1,23741.305346,614.790132,4013.251397,576.451327,286.171799,729.806549,1381.566243,1759.477327,503.881445,453.219451,...,214.971160,1263.811341,50.661993,412.142160,4515.763598,2272.943473,276.587097,M,Gut,133
B1_1,26292.108219,472.266501,1996.967953,731.856697,308.067883,889.800129,1341.737278,1681.081088,476.957890,440.990574,...,148.560654,1469.968579,10.946575,688.070399,3907.927106,1620.093030,320.578254,M,Gut,134
B2_1,22277.241433,401.416172,988.622938,595.343580,231.899208,809.613023,1475.475661,1647.704897,394.635494,561.440187,...,233.255343,1122.880374,21.698171,565.508594,3923.300631,1360.204125,259.021922,M,Gut,103
B4_1,21628.605092,513.719306,1576.939361,545.667024,249.192200,802.526676,1247.238911,1469.595028,609.562460,591.671738,...,154.626955,1594.830083,44.726805,522.664667,4895.668308,1828.687379,385.928434,M,Gut,47
C2_1,14676.825139,518.280406,1492.063592,346.006919,105.116026,617.556653,1084.738991,1257.012478,586.897812,543.099468,...,134.314922,1721.274927,21.899172,452.582890,3622.123065,1556.301164,243.810783,M,Gut,133


In [11]:
gut_sub_df_male = gut_df_male.iloc[:,:-3]
gut_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A2_1,23741.305346,614.790132,4013.251397,576.451327,286.171799,729.806549,1381.566243,1759.477327,503.881445,453.219451,...,1070.748070,116.385660,839.345993,214.971160,1263.811341,50.661993,412.142160,4515.763598,2272.943473,276.587097
B1_1,26292.108219,472.266501,1996.967953,731.856697,308.067883,889.800129,1341.737278,1681.081088,476.957890,440.990574,...,816.301700,129.795098,1013.340042,148.560654,1469.968579,10.946575,688.070399,3907.927106,1620.093030,320.578254
B2_1,22277.241433,401.416172,988.622938,595.343580,231.899208,809.613023,1475.475661,1647.704897,394.635494,561.440187,...,797.407802,153.243336,972.349310,233.255343,1122.880374,21.698171,565.508594,3923.300631,1360.204125,259.021922
B4_1,21628.605092,513.719306,1576.939361,545.667024,249.192200,802.526676,1247.238911,1469.595028,609.562460,591.671738,...,764.189415,116.289694,1166.730662,154.626955,1594.830083,44.726805,522.664667,4895.668308,1828.687379,385.928434
C2_1,14676.825139,518.280406,1492.063592,346.006919,105.116026,617.556653,1084.738991,1257.012478,586.897812,543.099468,...,819.029036,68.617406,858.447546,134.314922,1721.274927,21.899172,452.582890,3622.123065,1556.301164,243.810783


In [12]:
gut_sub_df_male["age"] = gut_df_male["age_days"].tolist()

gut_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A2_1,23741.305346,614.790132,4013.251397,576.451327,286.171799,729.806549,1381.566243,1759.477327,503.881445,453.219451,...,116.385660,839.345993,214.971160,1263.811341,50.661993,412.142160,4515.763598,2272.943473,276.587097,133
B1_1,26292.108219,472.266501,1996.967953,731.856697,308.067883,889.800129,1341.737278,1681.081088,476.957890,440.990574,...,129.795098,1013.340042,148.560654,1469.968579,10.946575,688.070399,3907.927106,1620.093030,320.578254,134
B2_1,22277.241433,401.416172,988.622938,595.343580,231.899208,809.613023,1475.475661,1647.704897,394.635494,561.440187,...,153.243336,972.349310,233.255343,1122.880374,21.698171,565.508594,3923.300631,1360.204125,259.021922,103
B4_1,21628.605092,513.719306,1576.939361,545.667024,249.192200,802.526676,1247.238911,1469.595028,609.562460,591.671738,...,116.289694,1166.730662,154.626955,1594.830083,44.726805,522.664667,4895.668308,1828.687379,385.928434,47
C2_1,14676.825139,518.280406,1492.063592,346.006919,105.116026,617.556653,1084.738991,1257.012478,586.897812,543.099468,...,68.617406,858.447546,134.314922,1721.274927,21.899172,452.582890,3622.123065,1556.301164,243.810783,133


In [13]:
gut_data_male = gut_sub_df_male.values

In [14]:
X, y = gut_data_male[:,:-1], gut_data_male[:,-1]

In [15]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.559e+00, tolerance: 5.403e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.554e+00, tolerance: 5.398e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.01, 'l1_ratio': 1.0}
Mean MAE: 13.666
Pearson correlation: 0.916
R-squared: 0.839


In [16]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 13.666
Pearson correlation: 0.916
R-squared: 0.839
Average number of non-zero coefficients: 50.34


#### Female

In [17]:
gut_df_female = gut_df.loc[gut_df.iloc[:,-3] == "F"]

gut_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A1_1,23876.707871,687.004459,2181.018965,576.907591,221.294706,884.077854,1368.504075,1348.686639,619.845370,636.359900,...,243.314079,1718.612117,47.341653,568.099841,3323.824460,1566.678439,358.915791,F,Gut,155
A3_1,25474.794219,394.841817,2380.296399,618.502214,223.660397,730.957162,1441.922333,1626.848248,689.723681,502.298768,...,252.398883,1593.111763,31.237486,541.033250,3708.514285,2226.607970,314.873854,F,Gut,134
A4_1,21094.343422,503.698033,2804.802732,581.903780,319.450595,775.429867,1249.966435,1181.039336,534.185019,514.302202,...,172.317748,1569.417030,86.158874,393.679779,3952.704039,1851.753033,279.684961,F,Gut,75
B3_1,22964.583416,519.967810,4104.173403,560.983083,284.460761,871.905310,1450.088346,1251.627350,931.443609,633.752115,...,197.137923,1903.902491,115.107378,633.752115,3274.606439,1966.086936,330.768327,F,Gut,52
C1_1,26337.165612,643.910106,4019.779650,526.929653,286.757395,961.724258,1432.751747,1300.242915,731.904253,672.896413,...,190.481446,2030.076717,49.690812,661.508935,3735.092706,2305.446634,364.399288,F,Gut,52


In [18]:
gut_sub_df_female = gut_df_female.iloc[:,:-3]

gut_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A1_1,23876.707871,687.004459,2181.018965,576.907591,221.294706,884.077854,1368.504075,1348.686639,619.845370,636.359900,...,913.804009,178.356927,1117.483215,243.314079,1718.612117,47.341653,568.099841,3323.824460,1566.678439,358.915791
A3_1,25474.794219,394.841817,2380.296399,618.502214,223.660397,730.957162,1441.922333,1626.848248,689.723681,502.298768,...,804.677628,153.688429,1270.740912,252.398883,1593.111763,31.237486,541.033250,3708.514285,2226.607970,314.873854
A4_1,21094.343422,503.698033,2804.802732,581.903780,319.450595,775.429867,1249.966435,1181.039336,534.185019,514.302202,...,770.127782,163.039100,1112.112236,172.317748,1569.417030,86.158874,393.679779,3952.704039,1851.753033,279.684961
B3_1,22964.583416,519.967810,4104.173403,560.983083,284.460761,871.905310,1450.088346,1251.627350,931.443609,633.752115,...,591.413769,140.245771,1239.719690,197.137923,1903.902491,115.107378,633.752115,3274.606439,1966.086936,330.768327
C1_1,26337.165612,643.910106,4019.779650,526.929653,286.757395,961.724258,1432.751747,1300.242915,731.904253,672.896413,...,800.229119,156.319013,1371.673458,190.481446,2030.076717,49.690812,661.508935,3735.092706,2305.446634,364.399288


In [19]:
gut_sub_df_female["age"] = gut_df_female["age_days"].tolist()

gut_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A1_1,23876.707871,687.004459,2181.018965,576.907591,221.294706,884.077854,1368.504075,1348.686639,619.845370,636.359900,...,178.356927,1117.483215,243.314079,1718.612117,47.341653,568.099841,3323.824460,1566.678439,358.915791,155
A3_1,25474.794219,394.841817,2380.296399,618.502214,223.660397,730.957162,1441.922333,1626.848248,689.723681,502.298768,...,153.688429,1270.740912,252.398883,1593.111763,31.237486,541.033250,3708.514285,2226.607970,314.873854,134
A4_1,21094.343422,503.698033,2804.802732,581.903780,319.450595,775.429867,1249.966435,1181.039336,534.185019,514.302202,...,163.039100,1112.112236,172.317748,1569.417030,86.158874,393.679779,3952.704039,1851.753033,279.684961,75
B3_1,22964.583416,519.967810,4104.173403,560.983083,284.460761,871.905310,1450.088346,1251.627350,931.443609,633.752115,...,140.245771,1239.719690,197.137923,1903.902491,115.107378,633.752115,3274.606439,1966.086936,330.768327,52
C1_1,26337.165612,643.910106,4019.779650,526.929653,286.757395,961.724258,1432.751747,1300.242915,731.904253,672.896413,...,156.319013,1371.673458,190.481446,2030.076717,49.690812,661.508935,3735.092706,2305.446634,364.399288,52


In [20]:
gut_data_female = gut_sub_df_female.values

In [21]:
X, y = gut_data_female[:,:-1], gut_data_female[:,-1]

In [22]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.156e+00, tolerance: 2.958e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.571e+00, tolerance: 3.281e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 1.0, 'l1_ratio': 1.0}
Mean MAE: 11.917
Pearson correlation: 0.921
R-squared: 0.835


In [23]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 11.917
Pearson correlation: 0.921
R-squared: 0.835
Average number of non-zero coefficients: 19.78


# Kidney

In [24]:
kidney_df = combined_df.loc[combined_df.iloc[:,-2] == "Kidney"]

kidney_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A10_1,4821.965545,907.596706,218.510659,554.051370,191.503723,684.175695,1125.288972,662.897504,586.787050,637.527352,...,112.938093,1122.015404,311.807344,542.593883,2612.307200,408.377598,391.191366,M,Kidney,133
A11_1,7011.583272,766.674115,222.103168,627.278822,272.285474,675.602523,1267.567871,975.767056,739.724358,713.703904,...,109.657631,1059.404232,398.670540,521.338398,1948.746206,360.569160,446.064940,M,Kidney,47
A12_1,7322.278018,732.665214,225.267215,592.693352,260.260180,651.743981,1162.787081,875.553156,586.132171,640.808680,...,111.540077,1065.827406,275.569603,626.957297,1945.025662,390.025761,450.534430,M,Kidney,161
A9_1,2831.666919,957.341072,263.063277,559.756802,399.078948,647.195448,1079.157306,842.998228,635.238026,610.575844,...,192.813424,1274.960085,330.323774,555.272769,3161.990693,289.220137,463.350090,F,Kidney,78
B10_1,6450.431017,785.062255,152.377915,537.184817,256.303867,585.636781,1329.971298,726.779458,588.445590,641.110768,...,100.414940,1010.469218,199.425475,639.706364,2482.987598,562.464102,502.776901,M,Kidney,147


In [25]:
kidney_df_male = kidney_df.loc[kidney_df.iloc[:,-3] == "M"]

kidney_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A10_1,4821.965545,907.596706,218.510659,554.051370,191.503723,684.175695,1125.288972,662.897504,586.787050,637.527352,...,112.938093,1122.015404,311.807344,542.593883,2612.307200,408.377598,391.191366,M,Kidney,133
A11_1,7011.583272,766.674115,222.103168,627.278822,272.285474,675.602523,1267.567871,975.767056,739.724358,713.703904,...,109.657631,1059.404232,398.670540,521.338398,1948.746206,360.569160,446.064940,M,Kidney,47
A12_1,7322.278018,732.665214,225.267215,592.693352,260.260180,651.743981,1162.787081,875.553156,586.132171,640.808680,...,111.540077,1065.827406,275.569603,626.957297,1945.025662,390.025761,450.534430,M,Kidney,161
B10_1,6450.431017,785.062255,152.377915,537.184817,256.303867,585.636781,1329.971298,726.779458,588.445590,641.110768,...,100.414940,1010.469218,199.425475,639.706364,2482.987598,562.464102,502.776901,M,Kidney,147
C10_1,2191.881729,1131.253113,245.924590,532.206035,206.828886,693.633459,1050.539402,646.970844,760.474501,645.709692,...,158.905120,1319.164723,321.593694,514.549911,3126.395170,524.639125,464.103841,M,Kidney,133


In [26]:
kidney_sub_df_male = kidney_df_male.iloc[:,:-3]
kidney_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A10_1,4821.965545,907.596706,218.510659,554.051370,191.503723,684.175695,1125.288972,662.897504,586.787050,637.527352,...,520.497299,128.487541,1351.165159,112.938093,1122.015404,311.807344,542.593883,2612.307200,408.377598,391.191366
A11_1,7011.583272,766.674115,222.103168,627.278822,272.285474,675.602523,1267.567871,975.767056,739.724358,713.703904,...,420.973787,192.365505,1129.101879,109.657631,1059.404232,398.670540,521.338398,1948.746206,360.569160,446.064940
A12_1,7322.278018,732.665214,225.267215,592.693352,260.260180,651.743981,1162.787081,875.553156,586.132171,640.808680,...,580.300010,119.559299,1335.564848,111.540077,1065.827406,275.569603,626.957297,1945.025662,390.025761,450.534430
B10_1,6450.431017,785.062255,152.377915,537.184817,256.303867,585.636781,1329.971298,726.779458,588.445590,641.110768,...,599.680828,100.414940,1393.871714,100.414940,1010.469218,199.425475,639.706364,2482.987598,562.464102,502.776901
C10_1,2191.881729,1131.253113,245.924590,532.206035,206.828886,693.633459,1050.539402,646.970844,760.474501,645.709692,...,663.365817,161.427423,1315.381268,158.905120,1319.164723,321.593694,514.549911,3126.395170,524.639125,464.103841


In [27]:
kidney_sub_df_male["age"] = kidney_df_male["age_days"].tolist()

kidney_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A10_1,4821.965545,907.596706,218.510659,554.051370,191.503723,684.175695,1125.288972,662.897504,586.787050,637.527352,...,128.487541,1351.165159,112.938093,1122.015404,311.807344,542.593883,2612.307200,408.377598,391.191366,133
A11_1,7011.583272,766.674115,222.103168,627.278822,272.285474,675.602523,1267.567871,975.767056,739.724358,713.703904,...,192.365505,1129.101879,109.657631,1059.404232,398.670540,521.338398,1948.746206,360.569160,446.064940,47
A12_1,7322.278018,732.665214,225.267215,592.693352,260.260180,651.743981,1162.787081,875.553156,586.132171,640.808680,...,119.559299,1335.564848,111.540077,1065.827406,275.569603,626.957297,1945.025662,390.025761,450.534430,161
B10_1,6450.431017,785.062255,152.377915,537.184817,256.303867,585.636781,1329.971298,726.779458,588.445590,641.110768,...,100.414940,1393.871714,100.414940,1010.469218,199.425475,639.706364,2482.987598,562.464102,502.776901,147
C10_1,2191.881729,1131.253113,245.924590,532.206035,206.828886,693.633459,1050.539402,646.970844,760.474501,645.709692,...,161.427423,1315.381268,158.905120,1319.164723,321.593694,514.549911,3126.395170,524.639125,464.103841,133


In [28]:
kidney_data_male = kidney_sub_df_male.values

In [29]:
X, y = kidney_data_male[:,:-1], kidney_data_male[:,-1]

In [30]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.449e+00, tolerance: 5.340e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.046e+00, tolerance: 5.040e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.001, 'l1_ratio': 0.6000000000000001}
Mean MAE: 14.304
Pearson correlation: 0.907
R-squared: 0.807


In [31]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 14.304
Pearson correlation: 0.907
R-squared: 0.807
Average number of non-zero coefficients: 430.97


##### Female

In [32]:
kidney_df_female = kidney_df.loc[kidney_df.iloc[:,-3] == "F"]

kidney_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A9_1,2831.666919,957.341072,263.063277,559.756802,399.078948,647.195448,1079.157306,842.998228,635.238026,610.575844,...,192.813424,1274.960085,330.323774,555.272769,3161.990693,289.220137,463.350090,F,Kidney,78
B11_1,4008.974389,1184.107667,249.565293,654.002665,358.418240,650.462732,954.896990,562.849384,624.798216,689.401998,...,162.836929,1531.906108,292.044492,551.344601,3534.623336,403.552389,492.935703,F,Kidney,147
B12_1,6201.808902,822.555707,243.324560,574.483351,321.663199,851.042485,986.354679,1042.141285,760.834355,710.982494,...,144.807787,1153.714498,188.724902,489.023018,1469.442951,261.128796,376.262856,F,Kidney,49
B9_1,4603.395213,711.466940,201.193520,595.561978,353.546584,571.506231,951.295448,789.465877,637.112813,629.823193,...,129.026279,1109.480208,112.260152,561.300762,3228.572826,403.844964,650.234130,F,Kidney,134
E10_1,3009.785292,659.286302,136.331738,548.123500,312.514292,565.601928,1059.891871,825.680936,600.558784,553.017460,...,146.119658,1038.917757,94.383511,543.229541,3874.617907,415.287448,378.233181,F,Kidney,103


In [33]:
kidney_sub_df_female = kidney_df_female.iloc[:,:-3]

kidney_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A9_1,2831.666919,957.341072,263.063277,559.756802,399.078948,647.195448,1079.157306,842.998228,635.238026,610.575844,...,550.788736,183.098019,1128.481670,192.813424,1274.960085,330.323774,555.272769,3161.990693,289.220137,463.350090
B11_1,4008.974389,1184.107667,249.565293,654.002665,358.418240,650.462732,954.896990,562.849384,624.798216,689.401998,...,632.763066,192.926361,1256.676299,162.836929,1531.906108,292.044492,551.344601,3534.623336,403.552389,492.935703
B12_1,6201.808902,822.555707,243.324560,574.483351,321.663199,851.042485,986.354679,1042.141285,760.834355,710.982494,...,441.545055,185.164055,1126.414669,144.807787,1153.714498,188.724902,489.023018,1469.442951,261.128796,376.262856
B9_1,4603.395213,711.466940,201.193520,595.561978,353.546584,571.506231,951.295448,789.465877,637.112813,629.823193,...,723.859295,126.110431,1219.553474,129.026279,1109.480208,112.260152,561.300762,3228.572826,403.844964,650.234130
E10_1,3009.785292,659.286302,136.331738,548.123500,312.514292,565.601928,1059.891871,825.680936,600.558784,553.017460,...,741.784482,106.967979,1510.136174,146.119658,1038.917757,94.383511,543.229541,3874.617907,415.287448,378.233181


In [34]:
kidney_sub_df_female["age"] = kidney_df_female["age_days"].tolist()

kidney_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A9_1,2831.666919,957.341072,263.063277,559.756802,399.078948,647.195448,1079.157306,842.998228,635.238026,610.575844,...,183.098019,1128.481670,192.813424,1274.960085,330.323774,555.272769,3161.990693,289.220137,463.350090,78
B11_1,4008.974389,1184.107667,249.565293,654.002665,358.418240,650.462732,954.896990,562.849384,624.798216,689.401998,...,192.926361,1256.676299,162.836929,1531.906108,292.044492,551.344601,3534.623336,403.552389,492.935703,147
B12_1,6201.808902,822.555707,243.324560,574.483351,321.663199,851.042485,986.354679,1042.141285,760.834355,710.982494,...,185.164055,1126.414669,144.807787,1153.714498,188.724902,489.023018,1469.442951,261.128796,376.262856,49
B9_1,4603.395213,711.466940,201.193520,595.561978,353.546584,571.506231,951.295448,789.465877,637.112813,629.823193,...,126.110431,1219.553474,129.026279,1109.480208,112.260152,561.300762,3228.572826,403.844964,650.234130,134
E10_1,3009.785292,659.286302,136.331738,548.123500,312.514292,565.601928,1059.891871,825.680936,600.558784,553.017460,...,106.967979,1510.136174,146.119658,1038.917757,94.383511,543.229541,3874.617907,415.287448,378.233181,103


In [35]:
kidney_data_female = kidney_sub_df_female.values

In [36]:
X, y = kidney_data_female[:,:-1], kidney_data_female[:,-1]

In [37]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.250e+00, tolerance: 3.050e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.368e+00, tolerance: 3.097e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.1, 'l1_ratio': 0.9}
Mean MAE: 22.839
Pearson correlation: 0.685
R-squared: 0.440


In [38]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 22.839
Pearson correlation: 0.685
R-squared: 0.440
Average number of non-zero coefficients: 100.87


# Muscle

In [39]:
muscle_df = combined_df.loc[combined_df.iloc[:,-2] == "Muscle"]

muscle_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A13_1,2.350903,453.724234,134.001458,712.323538,481.935067,1024.993606,886.290343,2868.101375,759.341594,702.919927,...,96.387013,883.939440,14.105417,1288.294716,1539.841312,298.564651,347.933609,M,Muscle,134
A14_1,1.880148,327.145692,174.853732,627.969318,370.389089,844.186298,789.662016,2120.806558,693.774486,667.452418,...,58.284577,889.309842,13.161034,1154.410662,1583.084328,454.995733,344.067021,M,Muscle,147
A15_1,3.992771,419.240920,139.746973,778.590280,395.284296,934.308337,527.045728,2367.713007,842.474611,491.110792,...,155.718056,914.344483,11.978312,1094.019163,1481.317918,235.573469,331.399966,M,Muscle,52
A16_1,1.631269,482.855494,84.825965,908.616589,363.772889,944.504497,885.778829,2233.206661,1295.227238,750.383538,...,138.657828,614.988248,8.156343,960.817183,1249.551718,107.663725,438.811243,F,Muscle,103
B13_1,1.771236,504.802247,99.189213,825.395955,480.004944,1013.146966,998.977078,2463.789213,981.264719,793.513708,...,123.986517,683.697078,8.856180,1225.695280,1712.785168,258.600449,356.018427,F,Muscle,155


In [40]:
muscle_df_male = muscle_df.loc[muscle_df.iloc[:,-3] == "M"]

muscle_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A13_1,2.350903,453.724234,134.001458,712.323538,481.935067,1024.993606,886.290343,2868.101375,759.341594,702.919927,...,96.387013,883.939440,14.105417,1288.294716,1539.841312,298.564651,347.933609,M,Muscle,134
A14_1,1.880148,327.145692,174.853732,627.969318,370.389089,844.186298,789.662016,2120.806558,693.774486,667.452418,...,58.284577,889.309842,13.161034,1154.410662,1583.084328,454.995733,344.067021,M,Muscle,147
A15_1,3.992771,419.240920,139.746973,778.590280,395.284296,934.308337,527.045728,2367.713007,842.474611,491.110792,...,155.718056,914.344483,11.978312,1094.019163,1481.317918,235.573469,331.399966,M,Muscle,52
B15_1,9.285661,508.058322,78.264859,967.035292,348.875558,1122.238487,1540.093242,1349.073926,3329.572815,1822.642649,...,423.160848,902.035663,19.897846,687.138932,1776.214343,244.080238,241.427192,M,Muscle,102
B16_1,2.358013,544.700927,77.814418,811.156359,261.739406,1436.029716,929.056992,1794.447642,1973.656605,1058.747689,...,134.406722,728.625915,21.222114,839.452511,1494.980033,162.702874,282.961520,M,Muscle,133


In [41]:
muscle_sub_df_male = muscle_df_male.iloc[:,:-3]
muscle_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A13_1,2.350903,453.724234,134.001458,712.323538,481.935067,1024.993606,886.290343,2868.101375,759.341594,702.919927,...,700.569024,51.719861,1036.748120,96.387013,883.939440,14.105417,1288.294716,1539.841312,298.564651,347.933609
A14_1,1.880148,327.145692,174.853732,627.969318,370.389089,844.186298,789.662016,2120.806558,693.774486,667.452418,...,755.819358,43.243396,960.755453,58.284577,889.309842,13.161034,1154.410662,1583.084328,454.995733,344.067021
A15_1,3.992771,419.240920,139.746973,778.590280,395.284296,934.308337,527.045728,2367.713007,842.474611,491.110792,...,642.836078,31.942165,914.344483,155.718056,914.344483,11.978312,1094.019163,1481.317918,235.573469,331.399966
B15_1,9.285661,508.058322,78.264859,967.035292,348.875558,1122.238487,1540.093242,1349.073926,3329.572815,1822.642649,...,202.958024,102.142274,883.464341,423.160848,902.035663,19.897846,687.138932,1776.214343,244.080238,241.427192
B16_1,2.358013,544.700927,77.814418,811.156359,261.739406,1436.029716,929.056992,1794.447642,1973.656605,1058.747689,...,323.047736,87.246469,483.392597,134.406722,728.625915,21.222114,839.452511,1494.980033,162.702874,282.961520


In [42]:
muscle_sub_df_male["age"] = muscle_df_male["age_days"].tolist()

muscle_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A13_1,2.350903,453.724234,134.001458,712.323538,481.935067,1024.993606,886.290343,2868.101375,759.341594,702.919927,...,51.719861,1036.748120,96.387013,883.939440,14.105417,1288.294716,1539.841312,298.564651,347.933609,134
A14_1,1.880148,327.145692,174.853732,627.969318,370.389089,844.186298,789.662016,2120.806558,693.774486,667.452418,...,43.243396,960.755453,58.284577,889.309842,13.161034,1154.410662,1583.084328,454.995733,344.067021,147
A15_1,3.992771,419.240920,139.746973,778.590280,395.284296,934.308337,527.045728,2367.713007,842.474611,491.110792,...,31.942165,914.344483,155.718056,914.344483,11.978312,1094.019163,1481.317918,235.573469,331.399966,52
B15_1,9.285661,508.058322,78.264859,967.035292,348.875558,1122.238487,1540.093242,1349.073926,3329.572815,1822.642649,...,102.142274,883.464341,423.160848,902.035663,19.897846,687.138932,1776.214343,244.080238,241.427192,102
B16_1,2.358013,544.700927,77.814418,811.156359,261.739406,1436.029716,929.056992,1794.447642,1973.656605,1058.747689,...,87.246469,483.392597,134.406722,728.625915,21.222114,839.452511,1494.980033,162.702874,282.961520,133


In [43]:
muscle_data_male = muscle_sub_df_male.values

In [44]:
X, y = muscle_data_male[:,:-1], muscle_data_male[:,-1]

In [45]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.537e+00, tolerance: 5.398e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.589e+00, tolerance: 5.398e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.1, 'l1_ratio': 1.0}
Mean MAE: 9.554
Pearson correlation: 0.952
R-squared: 0.900


In [ ]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

##### Female

In [46]:
muscle_df_female = muscle_df.loc[muscle_df.iloc[:,-3] == "F"]

muscle_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A16_1,1.631269,482.855494,84.825965,908.616589,363.772889,944.504497,885.778829,2233.206661,1295.227238,750.383538,...,138.657828,614.988248,8.156343,960.817183,1249.551718,107.663725,438.811243,F,Muscle,103
B13_1,1.771236,504.802247,99.189213,825.395955,480.004944,1013.146966,998.977078,2463.789213,981.264719,793.513708,...,123.986517,683.697078,8.856180,1225.695280,1712.785168,258.600449,356.018427,F,Muscle,155
B14_1,11.425531,536.999954,260.502106,539.285061,429.599964,1042.008422,767.795680,1631.565819,918.612688,431.885070,...,89.119141,966.599918,68.553186,902.616945,1544.731784,404.463795,379.327627,F,Muscle,77
C15_1,2.699865,448.177510,110.694445,693.865182,315.884149,1158.241879,674.966130,1803.509499,1241.937679,683.065724,...,126.893632,664.166672,29.698510,1347.232395,1892.605028,350.982388,348.282523,F,Muscle,134
D13_1,97.927266,349.422289,166.921476,750.033831,454.026414,750.033831,843.509857,2152.174227,934.760264,600.917313,...,129.085941,721.100775,28.933056,794.546225,1531.226337,344.971050,338.294191,F,Muscle,102


In [47]:
muscle_sub_df_female = muscle_df_female.iloc[:,:-3]

muscle_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A16_1,1.631269,482.855494,84.825965,908.616589,363.772889,944.504497,885.778829,2233.206661,1295.227238,750.383538,...,468.174077,40.781714,610.094442,138.657828,614.988248,8.156343,960.817183,1249.551718,107.663725,438.811243
B13_1,1.771236,504.802247,99.189213,825.395955,480.004944,1013.146966,998.977078,2463.789213,981.264719,793.513708,...,706.723146,83.248090,825.395955,123.986517,683.697078,8.856180,1225.695280,1712.785168,258.600449,356.018427
B14_1,11.425531,536.999954,260.502106,539.285061,429.599964,1042.008422,767.795680,1631.565819,918.612688,431.885070,...,530.144636,29.706380,792.931848,89.119141,966.599918,68.553186,902.616945,1544.731784,404.463795,379.327627
C15_1,2.699865,448.177510,110.694445,693.865182,315.884149,1158.241879,674.966130,1803.509499,1241.937679,683.065724,...,531.873310,35.098239,845.057595,126.893632,664.166672,29.698510,1347.232395,1892.605028,350.982388,348.282523
D13_1,97.927266,349.422289,166.921476,750.033831,454.026414,750.033831,843.509857,2152.174227,934.760264,600.917313,...,474.056991,57.866112,723.326395,129.085941,721.100775,28.933056,794.546225,1531.226337,344.971050,338.294191


In [48]:
muscle_sub_df_female["age"] = muscle_df_female["age_days"].tolist()

muscle_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A16_1,1.631269,482.855494,84.825965,908.616589,363.772889,944.504497,885.778829,2233.206661,1295.227238,750.383538,...,40.781714,610.094442,138.657828,614.988248,8.156343,960.817183,1249.551718,107.663725,438.811243,103
B13_1,1.771236,504.802247,99.189213,825.395955,480.004944,1013.146966,998.977078,2463.789213,981.264719,793.513708,...,83.248090,825.395955,123.986517,683.697078,8.856180,1225.695280,1712.785168,258.600449,356.018427,155
B14_1,11.425531,536.999954,260.502106,539.285061,429.599964,1042.008422,767.795680,1631.565819,918.612688,431.885070,...,29.706380,792.931848,89.119141,966.599918,68.553186,902.616945,1544.731784,404.463795,379.327627,77
C15_1,2.699865,448.177510,110.694445,693.865182,315.884149,1158.241879,674.966130,1803.509499,1241.937679,683.065724,...,35.098239,845.057595,126.893632,664.166672,29.698510,1347.232395,1892.605028,350.982388,348.282523,134
D13_1,97.927266,349.422289,166.921476,750.033831,454.026414,750.033831,843.509857,2152.174227,934.760264,600.917313,...,57.866112,723.326395,129.085941,721.100775,28.933056,794.546225,1531.226337,344.971050,338.294191,102


In [49]:
muscle_data_female = muscle_sub_df_female.values

In [50]:
X, y = muscle_data_female[:,:-1], muscle_data_female[:,-1]

In [51]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.444e+00, tolerance: 3.172e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.134e+00, tolerance: 2.958e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.001, 'l1_ratio': 0.0}
Mean MAE: 21.304
Pearson correlation: 0.755
R-squared: 0.570


In [52]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 21.304
Pearson correlation: 0.755
R-squared: 0.570
Average number of non-zero coefficients: 25118.96


# Spleen

In [53]:
spleen_df = combined_df.loc[combined_df.iloc[:,-2] == "Spleen"]

spleen_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A17_2,500.972933,610.959908,255.453620,361.183035,1160.894784,772.037607,1334.035571,796.873376,806.807683,725.204443,...,172.431194,1597.294718,83.022427,485.361878,1625.678454,108.567789,412.273759,F,Spleen,78
A18_2,127.857810,563.956610,89.154905,447.847896,407.071622,689.741050,2365.715045,958.588013,563.956610,796.865161,...,174.854194,1036.684945,17.278082,632.377817,4201.338522,395.322526,494.153157,M,Spleen,152
A19_2,13.175394,714.765109,185.553462,409.535155,658.769686,637.908646,1812.714586,883.849329,396.359761,586.305020,...,223.981693,1208.842374,45.015929,540.191142,3514.536274,254.724279,603.872212,F,Spleen,155
A20_2,263.871273,759.985412,123.802618,364.178503,591.903026,819.627549,1278.691271,659.678182,647.930488,984.095261,...,136.453980,1659.135811,187.963098,445.508690,2697.451196,178.022742,395.806909,M,Spleen,47
B17_2,116.199639,712.597787,111.299654,216.299328,492.798469,678.997891,1679.294784,643.298002,613.198096,769.997609,...,265.999174,1023.396821,80.499750,521.498380,10473.367471,505.398430,413.698715,M,Spleen,102


In [54]:
spleen_df_male = spleen_df.loc[spleen_df.iloc[:,-3] == "M"]

spleen_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A18_2,127.857810,563.956610,89.154905,447.847896,407.071622,689.741050,2365.715045,958.588013,563.956610,796.865161,...,174.854194,1036.684945,17.278082,632.377817,4201.338522,395.322526,494.153157,M,Spleen,152
A20_2,263.871273,759.985412,123.802618,364.178503,591.903026,819.627549,1278.691271,659.678182,647.930488,984.095261,...,136.453980,1659.135811,187.963098,445.508690,2697.451196,178.022742,395.806909,M,Spleen,47
B17_2,116.199639,712.597787,111.299654,216.299328,492.798469,678.997891,1679.294784,643.298002,613.198096,769.997609,...,265.999174,1023.396821,80.499750,521.498380,10473.367471,505.398430,413.698715,M,Spleen,102
B19_2,61.384651,568.164909,193.195103,390.672856,743.277712,683.796461,1412.798672,992.623116,659.528110,827.503163,...,165.595803,1333.807571,66.619001,568.164909,2992.144847,212.704953,549.130909,M,Spleen,78
B20_2,0.787620,631.671479,191.391732,389.872047,827.001312,581.263779,1487.814742,858.506124,671.052493,867.957568,...,207.144138,1233.413386,32.292432,562.360892,3421.422572,289.844269,515.891295,M,Spleen,161


In [55]:
spleen_sub_df_male = spleen_df_male.iloc[:,:-3]
spleen_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A18_2,127.857810,563.956610,89.154905,447.847896,407.071622,689.741050,2365.715045,958.588013,563.956610,796.865161,...,955.132396,143.062522,1398.833553,174.854194,1036.684945,17.278082,632.377817,4201.338522,395.322526,494.153157
A20_2,263.871273,759.985412,123.802618,364.178503,591.903026,819.627549,1278.691271,659.678182,647.930488,984.095261,...,611.783739,99.403562,1347.370095,136.453980,1659.135811,187.963098,445.508690,2697.451196,178.022742,395.806909
B17_2,116.199639,712.597787,111.299654,216.299328,492.798469,678.997891,1679.294784,643.298002,613.198096,769.997609,...,1406.295632,88.899724,1574.995108,265.999174,1023.396821,80.499750,521.498380,10473.367471,505.398430,413.698715
B19_2,61.384651,568.164909,193.195103,390.672856,743.277712,683.796461,1412.798672,992.623116,659.528110,827.503163,...,761.360012,116.583252,1200.093719,165.595803,1333.807571,66.619001,568.164909,2992.144847,212.704953,549.130909
B20_2,0.787620,631.671479,191.391732,389.872047,827.001312,581.263779,1487.814742,858.506124,671.052493,867.957568,...,887.648075,68.522966,1338.954506,207.144138,1233.413386,32.292432,562.360892,3421.422572,289.844269,515.891295


In [56]:
spleen_sub_df_male["age"] = spleen_df_male["age_days"].tolist()

spleen_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A18_2,127.857810,563.956610,89.154905,447.847896,407.071622,689.741050,2365.715045,958.588013,563.956610,796.865161,...,143.062522,1398.833553,174.854194,1036.684945,17.278082,632.377817,4201.338522,395.322526,494.153157,152
A20_2,263.871273,759.985412,123.802618,364.178503,591.903026,819.627549,1278.691271,659.678182,647.930488,984.095261,...,99.403562,1347.370095,136.453980,1659.135811,187.963098,445.508690,2697.451196,178.022742,395.806909,47
B17_2,116.199639,712.597787,111.299654,216.299328,492.798469,678.997891,1679.294784,643.298002,613.198096,769.997609,...,88.899724,1574.995108,265.999174,1023.396821,80.499750,521.498380,10473.367471,505.398430,413.698715,102
B19_2,61.384651,568.164909,193.195103,390.672856,743.277712,683.796461,1412.798672,992.623116,659.528110,827.503163,...,116.583252,1200.093719,165.595803,1333.807571,66.619001,568.164909,2992.144847,212.704953,549.130909,78
B20_2,0.787620,631.671479,191.391732,389.872047,827.001312,581.263779,1487.814742,858.506124,671.052493,867.957568,...,68.522966,1338.954506,207.144138,1233.413386,32.292432,562.360892,3421.422572,289.844269,515.891295,161


In [57]:
spleen_data_male = spleen_sub_df_male.values

In [58]:
X, y = spleen_data_male[:,:-1], spleen_data_male[:,-1]

In [59]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.396e+00, tolerance: 5.277e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.808e+00, tolerance: 5.448e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 1.0, 'l1_ratio': 1.0}
Mean MAE: 14.893
Pearson correlation: 0.866
R-squared: 0.745


In [60]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 14.893
Pearson correlation: 0.866
R-squared: 0.745
Average number of non-zero coefficients: 27.03


##### Female

In [61]:
spleen_df_female = spleen_df.loc[spleen_df.iloc[:,-3] == "F"]

spleen_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A17_2,500.972933,610.959908,255.453620,361.183035,1160.894784,772.037607,1334.035571,796.873376,806.807683,725.204443,...,172.431194,1597.294718,83.022427,485.361878,1625.678454,108.567789,412.273759,F,Spleen,78
A19_2,13.175394,714.765109,185.553462,409.535155,658.769686,637.908646,1812.714586,883.849329,396.359761,586.305020,...,223.981693,1208.842374,45.015929,540.191142,3514.536274,254.724279,603.872212,F,Spleen,155
B18_2,30.277472,820.712755,248.661792,376.213696,1014.617417,733.745548,1430.771609,709.910091,776.262850,963.725496,...,188.106848,1454.607066,75.371580,700.891270,3133.396266,237.066165,581.713986,F,Spleen,75
C18_2,4.621708,538.098877,171.663445,386.902996,921.700653,592.238886,1420.184888,1066.954337,457.549106,682.692317,...,198.073206,1071.576045,34.332689,839.830394,2272.559919,197.412962,644.398164,F,Spleen,75
C19_2,19.181350,903.542557,150.926941,559.792568,522.944184,732.424721,1566.308690,561.811657,773.816056,1001.468399,...,186.765780,1288.683882,111.049923,606.736399,3812.545774,393.722455,490.133979,F,Spleen,155


In [62]:
spleen_sub_df_female = spleen_df_female.iloc[:,:-3]

spleen_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A17_2,500.972933,610.959908,255.453620,361.183035,1160.894784,772.037607,1334.035571,796.873376,806.807683,725.204443,...,503.101713,137.661117,1245.336398,172.431194,1597.294718,83.022427,485.361878,1625.678454,108.567789,412.273759
A19_2,13.175394,714.765109,185.553462,409.535155,658.769686,637.908646,1812.714586,883.849329,396.359761,586.305020,...,1167.120293,96.619554,1352.673755,223.981693,1208.842374,45.015929,540.191142,3514.536274,254.724279,603.872212
B18_2,30.277472,820.712755,248.661792,376.213696,1014.617417,733.745548,1430.771609,709.910091,776.262850,963.725496,...,803.963516,100.495439,1523.536630,188.106848,1454.607066,75.371580,700.891270,3133.396266,237.066165,581.713986
C18_2,4.621708,538.098877,171.663445,386.902996,921.700653,592.238886,1420.184888,1066.954337,457.549106,682.692317,...,1180.516309,69.325622,1193.721189,198.073206,1071.576045,34.332689,839.830394,2272.559919,197.412962,644.398164
C19_2,19.181350,903.542557,150.926941,559.792568,522.944184,732.424721,1566.308690,561.811657,773.816056,1001.468399,...,727.881770,101.459248,1445.668092,186.765780,1288.683882,111.049923,606.736399,3812.545774,393.722455,490.133979


In [63]:
spleen_sub_df_female["age"] = spleen_df_female["age_days"].tolist()

spleen_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A17_2,500.972933,610.959908,255.453620,361.183035,1160.894784,772.037607,1334.035571,796.873376,806.807683,725.204443,...,137.661117,1245.336398,172.431194,1597.294718,83.022427,485.361878,1625.678454,108.567789,412.273759,78
A19_2,13.175394,714.765109,185.553462,409.535155,658.769686,637.908646,1812.714586,883.849329,396.359761,586.305020,...,96.619554,1352.673755,223.981693,1208.842374,45.015929,540.191142,3514.536274,254.724279,603.872212,155
B18_2,30.277472,820.712755,248.661792,376.213696,1014.617417,733.745548,1430.771609,709.910091,776.262850,963.725496,...,100.495439,1523.536630,188.106848,1454.607066,75.371580,700.891270,3133.396266,237.066165,581.713986,75
C18_2,4.621708,538.098877,171.663445,386.902996,921.700653,592.238886,1420.184888,1066.954337,457.549106,682.692317,...,69.325622,1193.721189,198.073206,1071.576045,34.332689,839.830394,2272.559919,197.412962,644.398164,75
C19_2,19.181350,903.542557,150.926941,559.792568,522.944184,732.424721,1566.308690,561.811657,773.816056,1001.468399,...,101.459248,1445.668092,186.765780,1288.683882,111.049923,606.736399,3812.545774,393.722455,490.133979,155


In [64]:
spleen_data_female = spleen_sub_df_female.values

In [65]:
X, y = spleen_data_female[:,:-1], spleen_data_female[:,-1]

In [66]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.574e+00, tolerance: 3.281e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.315e+00, tolerance: 3.097e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.01, 'l1_ratio': 0.9}
Mean MAE: 17.555
Pearson correlation: 0.831
R-squared: 0.681


In [67]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 17.555
Pearson correlation: 0.831
R-squared: 0.681
Average number of non-zero coefficients: 92.26


# Spinal Cord

In [68]:
spinal_df = combined_df.loc[combined_df.iloc[:,-2] == "SpinalCord"]

spinal_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A10_2,4.382338,451.380822,1198.256441,303.633424,289.860362,445.120339,1093.080327,468.284126,709.312721,492.073962,...,174.667475,1304.058604,28.172173,1264.617561,1144.416288,157.764171,238.524401,M,SpinalCord,75
A11_2,8.609836,486.229181,1020.945335,358.441083,187.603803,594.985009,1253.410917,559.639365,897.235581,560.545663,...,111.474724,1103.871654,52.565317,1282.865621,983.787094,216.605357,251.951001,F,SpinalCord,52
A12_2,6.381242,390.531984,1030.570512,342.672672,241.849055,465.830634,1290.287044,752.986504,714.699055,485.612483,...,134.644197,1193.930296,27.439339,1560.851686,1036.951754,135.282321,209.304723,M,SpinalCord,161
A9_2,9.612729,374.060560,1091.253769,354.835101,266.648757,382.837400,1291.449309,572.166377,649.904102,442.603501,...,180.552136,1248.818943,31.345857,1222.906368,1031.069723,128.309041,268.320536,M,SpinalCord,162
B10_2,12.443541,383.261073,1129.873553,309.429394,199.096661,482.809404,1082.588096,816.296311,808.830187,433.864808,...,176.698287,1089.224651,42.308041,1119.089150,905.060240,150.152065,199.926231,M,SpinalCord,103


In [69]:
spinal_df_male = spinal_df.loc[spinal_df.iloc[:,-3] == "M"]

spinal_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A10_2,4.382338,451.380822,1198.256441,303.633424,289.860362,445.120339,1093.080327,468.284126,709.312721,492.073962,...,174.667475,1304.058604,28.172173,1264.617561,1144.416288,157.764171,238.524401,M,SpinalCord,75
A12_2,6.381242,390.531984,1030.570512,342.672672,241.849055,465.830634,1290.287044,752.986504,714.699055,485.612483,...,134.644197,1193.930296,27.439339,1560.851686,1036.951754,135.282321,209.304723,M,SpinalCord,161
A9_2,9.612729,374.060560,1091.253769,354.835101,266.648757,382.837400,1291.449309,572.166377,649.904102,442.603501,...,180.552136,1248.818943,31.345857,1222.906368,1031.069723,128.309041,268.320536,M,SpinalCord,162
B10_2,12.443541,383.261073,1129.873553,309.429394,199.096661,482.809404,1082.588096,816.296311,808.830187,433.864808,...,176.698287,1089.224651,42.308041,1119.089150,905.060240,150.152065,199.926231,M,SpinalCord,103
B11_2,15.469734,515.451538,1179.412523,272.886108,267.317004,493.175121,1227.059304,534.015219,900.338521,549.484953,...,102.100245,1209.733201,59.403779,1386.088169,1094.019591,127.470608,202.344121,M,SpinalCord,52


In [70]:
spinal_sub_df_male = spinal_df_male.iloc[:,:-3]
spinal_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A10_2,4.382338,451.380822,1198.256441,303.633424,289.860362,445.120339,1093.080327,468.284126,709.312721,492.073962,...,1201.386683,131.470142,800.089724,174.667475,1304.058604,28.172173,1264.617561,1144.416288,157.764171,238.524401
A12_2,6.381242,390.531984,1030.570512,342.672672,241.849055,465.830634,1290.287044,752.986504,714.699055,485.612483,...,1118.631646,170.379150,608.132321,134.644197,1193.930296,27.439339,1560.851686,1036.951754,135.282321,209.304723
A9_2,9.612729,374.060560,1091.253769,354.835101,266.648757,382.837400,1291.449309,572.166377,649.904102,442.603501,...,916.134914,104.068245,540.820520,180.552136,1248.818943,31.345857,1222.906368,1031.069723,128.309041,268.320536
B10_2,12.443541,383.261073,1129.873553,309.429394,199.096661,482.809404,1082.588096,816.296311,808.830187,433.864808,...,1066.826277,227.302022,645.405010,176.698287,1089.224651,42.308041,1119.089150,905.060240,150.152065,199.926231
B11_2,15.469734,515.451538,1179.412523,272.886108,267.317004,493.175121,1227.059304,534.015219,900.338521,549.484953,...,1348.960808,157.791287,843.409900,102.100245,1209.733201,59.403779,1386.088169,1094.019591,127.470608,202.344121


In [71]:
spinal_sub_df_male["age"] = spinal_df_male["age_days"].tolist()

spinal_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A10_2,4.382338,451.380822,1198.256441,303.633424,289.860362,445.120339,1093.080327,468.284126,709.312721,492.073962,...,131.470142,800.089724,174.667475,1304.058604,28.172173,1264.617561,1144.416288,157.764171,238.524401,75
A12_2,6.381242,390.531984,1030.570512,342.672672,241.849055,465.830634,1290.287044,752.986504,714.699055,485.612483,...,170.379150,608.132321,134.644197,1193.930296,27.439339,1560.851686,1036.951754,135.282321,209.304723,161
A9_2,9.612729,374.060560,1091.253769,354.835101,266.648757,382.837400,1291.449309,572.166377,649.904102,442.603501,...,104.068245,540.820520,180.552136,1248.818943,31.345857,1222.906368,1031.069723,128.309041,268.320536,162
B10_2,12.443541,383.261073,1129.873553,309.429394,199.096661,482.809404,1082.588096,816.296311,808.830187,433.864808,...,227.302022,645.405010,176.698287,1089.224651,42.308041,1119.089150,905.060240,150.152065,199.926231,103
B11_2,15.469734,515.451538,1179.412523,272.886108,267.317004,493.175121,1227.059304,534.015219,900.338521,549.484953,...,157.791287,843.409900,102.100245,1209.733201,59.403779,1386.088169,1094.019591,127.470608,202.344121,52


In [72]:
spinal_data_male = spinal_sub_df_male.values

In [73]:
X, y = spinal_data_male[:,:-1], spinal_data_male[:,-1]

In [74]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.848e+00, tolerance: 4.895e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.690e+00, tolerance: 4.797e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.01, 'l1_ratio': 1.0}
Mean MAE: 17.174
Pearson correlation: 0.849
R-squared: 0.716


In [75]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 17.174
Pearson correlation: 0.849
R-squared: 0.716
Average number of non-zero coefficients: 47.35


#### Female

In [76]:
spinal_df_female = spinal_df.loc[spinal_df.iloc[:,-3] == "F"]

spinal_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A11_2,8.609836,486.229181,1020.945335,358.441083,187.603803,594.985009,1253.410917,559.639365,897.235581,560.545663,...,111.474724,1103.871654,52.565317,1282.865621,983.787094,216.605357,251.951001,F,SpinalCord,52
B9_2,8.454052,532.605295,1076.265888,349.217391,254.271881,541.059347,1214.131972,913.687960,784.926240,453.917577,...,145.019512,1064.560278,35.767144,1388.415512,1244.696623,152.823253,260.774998,F,SpinalCord,155
C11_2,14.798287,459.525768,1077.938412,244.561171,246.118886,485.228057,1359.884730,775.741805,767.953233,475.881770,...,165.896591,1099.746414,31.154289,1306.143581,954.878969,116.828585,183.031450,F,SpinalCord,78
C9_2,11.350211,386.273295,1097.309068,279.727770,209.795828,559.089405,1229.117965,495.015635,742.889589,471.582943,...,139.863885,1168.705554,33.684496,1383.627283,1273.054264,153.410911,252.267583,F,SpinalCord,78
D10_2,51.208117,460.873057,958.853445,225.612575,244.166241,464.583790,1148.100835,992.250043,849.757890,413.375672,...,112.064141,1041.231721,21.522252,1283.913668,1266.844296,218.933256,280.531426,F,SpinalCord,147


In [77]:
spinal_sub_df_female = spinal_df_female.iloc[:,:-3]

spinal_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A11_2,8.609836,486.229181,1020.945335,358.441083,187.603803,594.985009,1253.410917,559.639365,897.235581,560.545663,...,1109.762595,169.024683,675.192432,111.474724,1103.871654,52.565317,1282.865621,983.787094,216.605357,251.951001
B9_2,8.454052,532.605295,1076.265888,349.217391,254.271881,541.059347,1214.131972,913.687960,784.926240,453.917577,...,999.529106,206.799125,582.679297,145.019512,1064.560278,35.767144,1388.415512,1244.696623,152.823253,260.774998
C11_2,14.798287,459.525768,1077.938412,244.561171,246.118886,485.228057,1359.884730,775.741805,767.953233,475.881770,...,991.485259,295.965749,796.770950,165.896591,1099.746414,31.154289,1306.143581,954.878969,116.828585,183.031450
C9_2,11.350211,386.273295,1097.309068,279.727770,209.795828,559.089405,1229.117965,495.015635,742.889589,471.582943,...,1239.369768,155.607726,781.699986,139.863885,1168.705554,33.684496,1383.627283,1273.054264,153.410911,252.267583
D10_2,51.208117,460.873057,958.853445,225.612575,244.166241,464.583790,1148.100835,992.250043,849.757890,413.375672,...,872.764436,198.895297,575.905784,112.064141,1041.231721,21.522252,1283.913668,1266.844296,218.933256,280.531426


In [78]:
spinal_sub_df_female["age"] = spinal_df_female["age_days"].tolist()

spinal_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A11_2,8.609836,486.229181,1020.945335,358.441083,187.603803,594.985009,1253.410917,559.639365,897.235581,560.545663,...,169.024683,675.192432,111.474724,1103.871654,52.565317,1282.865621,983.787094,216.605357,251.951001,52
B9_2,8.454052,532.605295,1076.265888,349.217391,254.271881,541.059347,1214.131972,913.687960,784.926240,453.917577,...,206.799125,582.679297,145.019512,1064.560278,35.767144,1388.415512,1244.696623,152.823253,260.774998,155
C11_2,14.798287,459.525768,1077.938412,244.561171,246.118886,485.228057,1359.884730,775.741805,767.953233,475.881770,...,295.965749,796.770950,165.896591,1099.746414,31.154289,1306.143581,954.878969,116.828585,183.031450,78
C9_2,11.350211,386.273295,1097.309068,279.727770,209.795828,559.089405,1229.117965,495.015635,742.889589,471.582943,...,155.607726,781.699986,139.863885,1168.705554,33.684496,1383.627283,1273.054264,153.410911,252.267583,78
D10_2,51.208117,460.873057,958.853445,225.612575,244.166241,464.583790,1148.100835,992.250043,849.757890,413.375672,...,198.895297,575.905784,112.064141,1041.231721,21.522252,1283.913668,1266.844296,218.933256,280.531426,147


In [79]:
spinal_data_female = spinal_sub_df_female.values

In [80]:
X, y = spinal_data_female[:,:-1], spinal_data_female[:,-1]

In [81]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.608e+00, tolerance: 3.293e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.246e+00, tolerance: 3.050e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.001, 'l1_ratio': 0.8}
Mean MAE: 23.302
Pearson correlation: 0.659
R-squared: 0.377


In [82]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 23.302
Pearson correlation: 0.659
R-squared: 0.377
Average number of non-zero coefficients: 196.13


# Heart

In [83]:
heart_df = combined_df.loc[combined_df.iloc[:,-2] == "Heart"]

heart_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A13_2,26.477799,505.640544,194.739939,536.388955,640.591905,766.147919,1105.234567,997.615127,667.923826,527.847730,...,213.530635,1276.059075,9.395348,825.082374,3853.800902,227.196596,538.951323,M,Heart,162
A14_2,2.932010,680.226251,240.424796,600.084653,488.668284,815.098698,1014.475358,1355.565820,690.976954,632.336760,...,98.710993,1270.537539,28.342760,699.772983,5555.181053,215.014045,434.914773,M,Heart,103
A15_2,0.837248,596.957947,225.219758,452.114013,365.040203,632.122370,1239.127294,843.108909,613.702910,489.790181,...,151.541919,1415.786659,79.538576,1085.910879,6306.153230,280.478138,380.947918,M,Heart,103
B13_2,20.962146,585.707020,218.252932,487.061627,445.137335,694.216952,912.469884,1789.180813,537.617391,647.360391,...,191.125449,1464.884084,53.021899,822.455963,4199.827601,228.117471,453.768807,M,Heart,134
B14_2,79.955588,662.489160,237.851078,465.623720,548.938787,626.878688,1090.486721,934.606918,580.517885,624.191105,...,176.708569,1688.473894,112.878478,876.151993,4780.537904,313.775292,423.966187,M,Heart,47


In [84]:
heart_df_male = heart_df.loc[heart_df.iloc[:,-3] == "M"]

heart_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A13_2,26.477799,505.640544,194.739939,536.388955,640.591905,766.147919,1105.234567,997.615127,667.923826,527.847730,...,213.530635,1276.059075,9.395348,825.082374,3853.800902,227.196596,538.951323,M,Heart,162
A14_2,2.932010,680.226251,240.424796,600.084653,488.668284,815.098698,1014.475358,1355.565820,690.976954,632.336760,...,98.710993,1270.537539,28.342760,699.772983,5555.181053,215.014045,434.914773,M,Heart,103
A15_2,0.837248,596.957947,225.219758,452.114013,365.040203,632.122370,1239.127294,843.108909,613.702910,489.790181,...,151.541919,1415.786659,79.538576,1085.910879,6306.153230,280.478138,380.947918,M,Heart,103
B13_2,20.962146,585.707020,218.252932,487.061627,445.137335,694.216952,912.469884,1789.180813,537.617391,647.360391,...,191.125449,1464.884084,53.021899,822.455963,4199.827601,228.117471,453.768807,M,Heart,134
B14_2,79.955588,662.489160,237.851078,465.623720,548.938787,626.878688,1090.486721,934.606918,580.517885,624.191105,...,176.708569,1688.473894,112.878478,876.151993,4780.537904,313.775292,423.966187,M,Heart,47


In [85]:
heart_sub_df_male = heart_df_male.iloc[:,:-3]
heart_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A13_2,26.477799,505.640544,194.739939,536.388955,640.591905,766.147919,1105.234567,997.615127,667.923826,527.847730,...,837.894212,68.329803,1483.610852,213.530635,1276.059075,9.395348,825.082374,3853.800902,227.196596,538.951323
A14_2,2.932010,680.226251,240.424796,600.084653,488.668284,815.098698,1014.475358,1355.565820,690.976954,632.336760,...,948.993808,102.620340,1423.979380,98.710993,1270.537539,28.342760,699.772983,5555.181053,215.014045,434.914773
A15_2,0.837248,596.957947,225.219758,452.114013,365.040203,632.122370,1239.127294,843.108909,613.702910,489.790181,...,764.407581,66.979854,1732.266467,151.541919,1415.786659,79.538576,1085.910879,6306.153230,280.478138,380.947918
B13_2,20.962146,585.707020,218.252932,487.061627,445.137335,694.216952,912.469884,1789.180813,537.617391,647.360391,...,766.967930,67.818708,1583.258555,191.125449,1464.884084,53.021899,822.455963,4199.827601,228.117471,453.768807
B14_2,79.955588,662.489160,237.851078,465.623720,548.938787,626.878688,1090.486721,934.606918,580.517885,624.191105,...,651.066933,81.971275,2270.335570,176.708569,1688.473894,112.878478,876.151993,4780.537904,313.775292,423.966187


In [86]:
heart_sub_df_male["age"] = heart_df_male["age_days"].tolist()

heart_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A13_2,26.477799,505.640544,194.739939,536.388955,640.591905,766.147919,1105.234567,997.615127,667.923826,527.847730,...,68.329803,1483.610852,213.530635,1276.059075,9.395348,825.082374,3853.800902,227.196596,538.951323,162
A14_2,2.932010,680.226251,240.424796,600.084653,488.668284,815.098698,1014.475358,1355.565820,690.976954,632.336760,...,102.620340,1423.979380,98.710993,1270.537539,28.342760,699.772983,5555.181053,215.014045,434.914773,103
A15_2,0.837248,596.957947,225.219758,452.114013,365.040203,632.122370,1239.127294,843.108909,613.702910,489.790181,...,66.979854,1732.266467,151.541919,1415.786659,79.538576,1085.910879,6306.153230,280.478138,380.947918,103
B13_2,20.962146,585.707020,218.252932,487.061627,445.137335,694.216952,912.469884,1789.180813,537.617391,647.360391,...,67.818708,1583.258555,191.125449,1464.884084,53.021899,822.455963,4199.827601,228.117471,453.768807,134
B14_2,79.955588,662.489160,237.851078,465.623720,548.938787,626.878688,1090.486721,934.606918,580.517885,624.191105,...,81.971275,2270.335570,176.708569,1688.473894,112.878478,876.151993,4780.537904,313.775292,423.966187,47


In [87]:
heart_data_male = heart_sub_df_male.values

In [88]:
X, y = heart_data_male[:,:-1], heart_data_male[:,-1]

In [89]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.276e+00, tolerance: 5.181e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.621e+00, tolerance: 5.450e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.01, 'l1_ratio': 0.0}
Mean MAE: 29.165
Pearson correlation: 0.343
R-squared: -0.932


In [90]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 29.165
Pearson correlation: 0.343
R-squared: -0.932
Average number of non-zero coefficients: 25122.00


##### Female

In [91]:
heart_df_female = heart_df.loc[heart_df.iloc[:,-3] == "F"]

heart_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
B15_2,1.775166,827.819273,272.783906,544.976091,801.783499,675.154961,985.217363,905.926596,734.327175,689.356293,...,155.622923,1545.578229,110.652040,766.871893,4251.523574,361.542227,383.435947,F,Heart,75
B16_2,0.755241,513.563623,179.747268,517.339826,587.577203,718.233831,1095.098901,1731.766745,589.842925,611.744903,...,124.614703,1042.232058,9.818128,783.939765,6263.965714,229.593149,460.696779,F,Heart,103
C15_2,0.701235,823.951607,310.647287,445.985721,725.077414,732.089768,1006.974049,774.163893,520.316674,619.190867,...,178.113794,1884.920782,164.790321,746.114476,5004.717120,399.002948,388.484417,F,Heart,75
D14_2,23.259273,634.753062,177.820893,513.954902,545.467466,843.336219,969.386473,1786.462222,642.256053,520.707594,...,101.290382,1246.246851,3.751496,776.559597,5473.432134,387.904649,558.222551,F,Heart,133
D16_2,2.694510,489.502624,433.816087,468.844715,573.930599,760.749949,1014.033875,1009.543026,689.794523,526.327592,...,135.623663,1711.013758,52.093857,787.695048,10794.206478,377.231380,522.734912,F,Heart,47


In [92]:
heart_sub_df_female = heart_df_female.iloc[:,:-3]

heart_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
B15_2,1.775166,827.819273,272.783906,544.976091,801.783499,675.154961,985.217363,905.926596,734.327175,689.356293,...,485.212155,89.350043,1805.344248,155.622923,1545.578229,110.652040,766.871893,4251.523574,361.542227,383.435947
B16_2,0.755241,513.563623,179.747268,517.339826,587.577203,718.233831,1095.098901,1731.766745,589.842925,611.744903,...,784.695006,129.146146,1262.762319,124.614703,1042.232058,9.818128,783.939765,6263.965714,229.593149,460.696779
C15_2,0.701235,823.951607,310.647287,445.985721,725.077414,732.089768,1006.974049,774.163893,520.316674,619.190867,...,685.106995,63.111187,2020.960451,178.113794,1884.920782,164.790321,746.114476,5004.717120,399.002948,388.484417
D14_2,23.259273,634.753062,177.820893,513.954902,545.467466,843.336219,969.386473,1786.462222,642.256053,520.707594,...,566.475841,120.047860,1852.488545,101.290382,1246.246851,3.751496,776.559597,5473.432134,387.904649,558.222551
D16_2,2.694510,489.502624,433.816087,468.844715,573.930599,760.749949,1014.033875,1009.543026,689.794523,526.327592,...,840.687075,48.501177,2074.772588,135.623663,1711.013758,52.093857,787.695048,10794.206478,377.231380,522.734912


In [93]:
heart_sub_df_female["age"] = heart_df_female["age_days"].tolist()

heart_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
B15_2,1.775166,827.819273,272.783906,544.976091,801.783499,675.154961,985.217363,905.926596,734.327175,689.356293,...,89.350043,1805.344248,155.622923,1545.578229,110.652040,766.871893,4251.523574,361.542227,383.435947,75
B16_2,0.755241,513.563623,179.747268,517.339826,587.577203,718.233831,1095.098901,1731.766745,589.842925,611.744903,...,129.146146,1262.762319,124.614703,1042.232058,9.818128,783.939765,6263.965714,229.593149,460.696779,103
C15_2,0.701235,823.951607,310.647287,445.985721,725.077414,732.089768,1006.974049,774.163893,520.316674,619.190867,...,63.111187,2020.960451,178.113794,1884.920782,164.790321,746.114476,5004.717120,399.002948,388.484417,75
D14_2,23.259273,634.753062,177.820893,513.954902,545.467466,843.336219,969.386473,1786.462222,642.256053,520.707594,...,120.047860,1852.488545,101.290382,1246.246851,3.751496,776.559597,5473.432134,387.904649,558.222551,133
D16_2,2.694510,489.502624,433.816087,468.844715,573.930599,760.749949,1014.033875,1009.543026,689.794523,526.327592,...,48.501177,2074.772588,135.623663,1711.013758,52.093857,787.695048,10794.206478,377.231380,522.734912,47


In [94]:
heart_data_female = heart_sub_df_female.values

In [95]:
X, y = heart_data_female[:,:-1], heart_data_female[:,-1]

In [96]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.962e+00, tolerance: 2.827e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.256e+00, tolerance: 3.043e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.0001, 'l1_ratio': 0.0}
Mean MAE: 21.860
Pearson correlation: 0.702
R-squared: 0.443


In [97]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 21.860
Pearson correlation: 0.702
R-squared: 0.443
Average number of non-zero coefficients: 24832.05


# Skin

In [98]:
skin_df = combined_df.loc[combined_df.iloc[:,-2] == "Skin"]

skin_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A17_1,0.943982,830.704200,302.074255,989.293184,623.972132,514.470215,865.631536,1193.193305,688.162911,543.733658,...,151.037127,1022.332555,89.678294,600.372581,6715.488271,527.685963,438.951651,F,Skin,134
A18_1,0.967501,645.323368,292.185393,921.061238,710.145955,565.988261,1256.784189,1532.522060,548.573237,582.435783,...,104.490140,650.160874,11.610016,700.470942,5032.941766,475.043139,543.735731,M,Skin,161
A19_1,7.352180,664.637032,227.917566,1073.418216,801.387572,460.246441,949.901599,1146.940012,571.999570,613.171775,...,77.933103,754.333623,35.290462,673.459648,5843.512316,388.195081,408.781184,M,Skin,162
A20_1,0.834110,719.837185,194.347699,1064.324736,759.874479,596.388861,1051.813082,1181.934288,535.498809,754.035707,...,76.738147,903.341449,41.705515,628.085052,6622.835742,518.816603,489.622743,M,Skin,161
B17_1,3.130785,817.134895,332.906809,948.627867,632.418578,578.151637,1076.990053,1243.965255,782.696260,675.205973,...,72.008056,1038.377038,42.787396,566.672092,6022.586819,658.508453,479.010111,F,Skin,155


In [99]:
skin_df_male = skin_df.loc[skin_df.iloc[:,-3] == "M"]

skin_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A18_1,0.967501,645.323368,292.185393,921.061238,710.145955,565.988261,1256.784189,1532.522060,548.573237,582.435783,...,104.490140,650.160874,11.610016,700.470942,5032.941766,475.043139,543.735731,M,Skin,161
A19_1,7.352180,664.637032,227.917566,1073.418216,801.387572,460.246441,949.901599,1146.940012,571.999570,613.171775,...,77.933103,754.333623,35.290462,673.459648,5843.512316,388.195081,408.781184,M,Skin,162
A20_1,0.834110,719.837185,194.347699,1064.324736,759.874479,596.388861,1051.813082,1181.934288,535.498809,754.035707,...,76.738147,903.341449,41.705515,628.085052,6622.835742,518.816603,489.622743,M,Skin,161
B19_1,1.054980,726.881356,318.604020,881.963446,790.180168,550.699663,964.251901,1141.488574,618.218396,705.781752,...,133.982485,983.241545,30.594426,699.451871,5694.783109,648.812822,489.510812,M,Skin,162
B20_1,1.093179,933.575229,224.101782,859.239028,678.864423,470.067153,947.786561,731.337035,701.821191,605.621401,...,150.858761,1432.065046,221.915423,537.844277,4675.528399,616.553195,382.612799,M,Skin,47


In [100]:
skin_sub_df_male = skin_df_male.iloc[:,:-3]
skin_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A18_1,0.967501,645.323368,292.185393,921.061238,710.145955,565.988261,1256.784189,1532.522060,548.573237,582.435783,...,1026.518880,85.140114,1407.714392,104.490140,650.160874,11.610016,700.470942,5032.941766,475.043139,543.735731
A19_1,7.352180,664.637032,227.917566,1073.418216,801.387572,460.246441,949.901599,1146.940012,571.999570,613.171775,...,552.883903,110.282693,1277.808808,77.933103,754.333623,35.290462,673.459648,5843.512316,388.195081,408.781184
A20_1,0.834110,719.837185,194.347699,1064.324736,759.874479,596.388861,1051.813082,1181.934288,535.498809,754.035707,...,882.488692,85.079250,1377.116097,76.738147,903.341449,41.705515,628.085052,6622.835742,518.816603,489.622743
B19_1,1.054980,726.881356,318.604020,881.963446,790.180168,550.699663,964.251901,1141.488574,618.218396,705.781752,...,707.891713,98.113158,1342.989792,133.982485,983.241545,30.594426,699.451871,5694.783109,648.812822,489.510812
B20_1,1.093179,933.575229,224.101782,859.239028,678.864423,470.067153,947.786561,731.337035,701.821191,605.621401,...,815.511851,120.249737,1562.153398,150.858761,1432.065046,221.915423,537.844277,4675.528399,616.553195,382.612799


In [101]:
skin_sub_df_male["age"] = skin_df_male["age_days"].tolist()

skin_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A18_1,0.967501,645.323368,292.185393,921.061238,710.145955,565.988261,1256.784189,1532.522060,548.573237,582.435783,...,85.140114,1407.714392,104.490140,650.160874,11.610016,700.470942,5032.941766,475.043139,543.735731,161
A19_1,7.352180,664.637032,227.917566,1073.418216,801.387572,460.246441,949.901599,1146.940012,571.999570,613.171775,...,110.282693,1277.808808,77.933103,754.333623,35.290462,673.459648,5843.512316,388.195081,408.781184,162
A20_1,0.834110,719.837185,194.347699,1064.324736,759.874479,596.388861,1051.813082,1181.934288,535.498809,754.035707,...,85.079250,1377.116097,76.738147,903.341449,41.705515,628.085052,6622.835742,518.816603,489.622743,161
B19_1,1.054980,726.881356,318.604020,881.963446,790.180168,550.699663,964.251901,1141.488574,618.218396,705.781752,...,98.113158,1342.989792,133.982485,983.241545,30.594426,699.451871,5694.783109,648.812822,489.510812,162
B20_1,1.093179,933.575229,224.101782,859.239028,678.864423,470.067153,947.786561,731.337035,701.821191,605.621401,...,120.249737,1562.153398,150.858761,1432.065046,221.915423,537.844277,4675.528399,616.553195,382.612799,47


In [102]:
skin_data_male = skin_sub_df_male.values

In [103]:
X, y = skin_data_male[:,:-1], skin_data_male[:,-1]

In [104]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.246e+00, tolerance: 5.192e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.423e+00, tolerance: 5.326e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.1, 'l1_ratio': 1.0}
Mean MAE: 9.996
Pearson correlation: 0.952
R-squared: 0.904


In [105]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 9.996
Pearson correlation: 0.952
R-squared: 0.904
Average number of non-zero coefficients: 34.34


##### Female

In [106]:
skin_df_female = skin_df.loc[skin_df.iloc[:,-3] == "F"]

skin_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A17_1,0.943982,830.704200,302.074255,989.293184,623.972132,514.470215,865.631536,1193.193305,688.162911,543.733658,...,151.037127,1022.332555,89.678294,600.372581,6715.488271,527.685963,438.951651,F,Skin,134
B17_1,3.130785,817.134895,332.906809,948.627867,632.418578,578.151637,1076.990053,1243.965255,782.696260,675.205973,...,72.008056,1038.377038,42.787396,566.672092,6022.586819,658.508453,479.010111,F,Skin,155
B18_1,1.035014,822.835943,251.508345,1106.429715,647.918617,590.992860,961.527788,1397.268583,721.404594,600.307984,...,102.466363,1029.838696,34.155454,662.408810,6771.060052,574.432640,495.771594,F,Skin,147
C17_1,1.221560,648.648196,196.671110,961.367477,555.809659,527.713786,862.421142,886.852336,480.072958,641.318838,...,172.239916,955.259678,73.293581,564.360577,5100.011709,421.438093,434.875250,F,Skin,52
C19_1,7.151353,649.342851,238.855190,1230.032713,636.470415,444.814155,1184.264054,2238.373483,307.508178,473.419567,...,127.294083,699.402322,34.326494,729.438004,5937.053246,430.511450,487.722273,F,Skin,133


In [107]:
skin_sub_df_female = skin_df_female.iloc[:,:-3]

skin_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A17_1,0.943982,830.704200,302.074255,989.293184,623.972132,514.470215,865.631536,1193.193305,688.162911,543.733658,...,804.272703,104.782007,1523.587021,151.037127,1022.332555,89.678294,600.372581,6715.488271,527.685963,438.951651
B17_1,3.130785,817.134895,332.906809,948.627867,632.418578,578.151637,1076.990053,1243.965255,782.696260,675.205973,...,1063.423318,80.356816,1230.398520,72.008056,1038.377038,42.787396,566.672092,6022.586819,658.508453,479.010111
B18_1,1.035014,822.835943,251.508345,1106.429715,647.918617,590.992860,961.527788,1397.268583,721.404594,600.307984,...,751.419994,101.431349,1400.373624,102.466363,1029.838696,34.155454,662.408810,6771.060052,574.432640,495.771594
C17_1,1.221560,648.648196,196.671110,961.367477,555.809659,527.713786,862.421142,886.852336,480.072958,641.318838,...,988.241790,97.724775,1349.823458,172.239916,955.259678,73.293581,564.360577,5100.011709,421.438093,434.875250
C19_1,7.151353,649.342851,238.855190,1230.032713,636.470415,444.814155,1184.264054,2238.373483,307.508178,473.419567,...,1018.352665,127.294083,1473.178714,127.294083,699.402322,34.326494,729.438004,5937.053246,430.511450,487.722273


In [108]:
skin_sub_df_female["age"] = skin_df_female["age_days"].tolist()

skin_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A17_1,0.943982,830.704200,302.074255,989.293184,623.972132,514.470215,865.631536,1193.193305,688.162911,543.733658,...,104.782007,1523.587021,151.037127,1022.332555,89.678294,600.372581,6715.488271,527.685963,438.951651,134
B17_1,3.130785,817.134895,332.906809,948.627867,632.418578,578.151637,1076.990053,1243.965255,782.696260,675.205973,...,80.356816,1230.398520,72.008056,1038.377038,42.787396,566.672092,6022.586819,658.508453,479.010111,155
B18_1,1.035014,822.835943,251.508345,1106.429715,647.918617,590.992860,961.527788,1397.268583,721.404594,600.307984,...,101.431349,1400.373624,102.466363,1029.838696,34.155454,662.408810,6771.060052,574.432640,495.771594,147
C17_1,1.221560,648.648196,196.671110,961.367477,555.809659,527.713786,862.421142,886.852336,480.072958,641.318838,...,97.724775,1349.823458,172.239916,955.259678,73.293581,564.360577,5100.011709,421.438093,434.875250,52
C19_1,7.151353,649.342851,238.855190,1230.032713,636.470415,444.814155,1184.264054,2238.373483,307.508178,473.419567,...,127.294083,1473.178714,127.294083,699.402322,34.326494,729.438004,5937.053246,430.511450,487.722273,133


In [109]:
skin_data_female = skin_sub_df_female.values

In [110]:
X, y = skin_data_female[:,:-1], skin_data_female[:,-1]

In [111]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.439e+00, tolerance: 3.150e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.089e+00, tolerance: 2.931e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.01, 'l1_ratio': 1.0}
Mean MAE: 15.105
Pearson correlation: 0.901
R-squared: 0.802


In [112]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 15.105
Pearson correlation: 0.901
R-squared: 0.802
Average number of non-zero coefficients: 38.32


# Brain

In [113]:
brain_df = combined_df.loc[combined_df.iloc[:,-2] == "Brain"]

brain_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A21_2,53.082851,503.745418,485.328919,309.830515,144.623685,546.536696,2395.769875,326.080367,486.953904,418.162863,...,215.039711,609.369457,19.499823,1015.074101,613.702751,159.790213,334.205294,M,Brain,162
A22_2,52.450281,430.841595,533.066122,344.673276,94.731630,475.263772,2237.165051,269.209096,439.404906,449.573838,...,261.180992,611.741544,21.408278,1192.976291,894.866020,181.435156,439.404906,M,Brain,152
A23_2,17.414194,584.876719,587.879166,315.256959,104.485163,422.744569,3009.653097,232.389415,351.286325,369.301009,...,210.771795,505.011623,31.225451,1231.003362,736.800549,254.007035,441.960231,F,Brain,78
A24_2,43.256835,536.384749,414.112096,325.868154,98.048825,484.476547,1956.939196,307.411904,321.254091,415.265612,...,223.205266,566.376154,23.647070,981.065008,760.166773,155.147847,410.651550,M,Brain,103
B21_2,40.582672,504.195584,454.790592,391.711005,112.043463,464.054028,1794.018758,178.651979,393.475469,470.670768,...,235.114826,775.040806,24.261380,1051.179420,1012.802328,187.474299,307.016734,M,Brain,162


In [114]:
brain_df_male = brain_df.loc[brain_df.iloc[:,-3] == "M"]

brain_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A21_2,53.082851,503.745418,485.328919,309.830515,144.623685,546.536696,2395.769875,326.080367,486.953904,418.162863,...,215.039711,609.369457,19.499823,1015.074101,613.702751,159.790213,334.205294,M,Brain,162
A22_2,52.450281,430.841595,533.066122,344.673276,94.731630,475.263772,2237.165051,269.209096,439.404906,449.573838,...,261.180992,611.741544,21.408278,1192.976291,894.866020,181.435156,439.404906,M,Brain,152
A24_2,43.256835,536.384749,414.112096,325.868154,98.048825,484.476547,1956.939196,307.411904,321.254091,415.265612,...,223.205266,566.376154,23.647070,981.065008,760.166773,155.147847,410.651550,M,Brain,103
B21_2,40.582672,504.195584,454.790592,391.711005,112.043463,464.054028,1794.018758,178.651979,393.475469,470.670768,...,235.114826,775.040806,24.261380,1051.179420,1012.802328,187.474299,307.016734,M,Brain,162
B22_2,46.363146,593.237531,411.999777,362.475507,83.769776,404.623822,2161.681696,285.027979,385.130227,367.217193,...,190.194271,613.784834,23.708427,1295.006973,913.564724,285.027979,431.493373,M,Brain,49


In [115]:
brain_sub_df_male = brain_df_male.iloc[:,:-3]
brain_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A21_2,53.082851,503.745418,485.328919,309.830515,144.623685,546.536696,2395.769875,326.080367,486.953904,418.162863,...,1468.986639,102.915731,523.786902,215.039711,609.369457,19.499823,1015.074101,613.702751,159.790213,334.205294
A22_2,52.450281,430.841595,533.066122,344.673276,94.731630,475.263772,2237.165051,269.209096,439.404906,449.573838,...,1841.111908,63.689627,754.641799,261.180992,611.741544,21.408278,1192.976291,894.866020,181.435156,439.404906
A24_2,43.256835,536.384749,414.112096,325.868154,98.048825,484.476547,1956.939196,307.411904,321.254091,415.265612,...,1837.550333,73.824998,694.416384,223.205266,566.376154,23.647070,981.065008,760.166773,155.147847,410.651550
B21_2,40.582672,504.195584,454.790592,391.711005,112.043463,464.054028,1794.018758,178.651979,393.475469,470.670768,...,1570.372948,43.229368,715.049030,235.114826,775.040806,24.261380,1051.179420,1012.802328,187.474299,307.016734
B22_2,46.363146,593.237531,411.999777,362.475507,83.769776,404.623822,2161.681696,285.027979,385.130227,367.217193,...,2326.060124,77.974382,936.746297,190.194271,613.784834,23.708427,1295.006973,913.564724,285.027979,431.493373


In [116]:
brain_sub_df_male["age"] = brain_df_male["age_days"].tolist()

brain_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A21_2,53.082851,503.745418,485.328919,309.830515,144.623685,546.536696,2395.769875,326.080367,486.953904,418.162863,...,102.915731,523.786902,215.039711,609.369457,19.499823,1015.074101,613.702751,159.790213,334.205294,162
A22_2,52.450281,430.841595,533.066122,344.673276,94.731630,475.263772,2237.165051,269.209096,439.404906,449.573838,...,63.689627,754.641799,261.180992,611.741544,21.408278,1192.976291,894.866020,181.435156,439.404906,152
A24_2,43.256835,536.384749,414.112096,325.868154,98.048825,484.476547,1956.939196,307.411904,321.254091,415.265612,...,73.824998,694.416384,223.205266,566.376154,23.647070,981.065008,760.166773,155.147847,410.651550,103
B21_2,40.582672,504.195584,454.790592,391.711005,112.043463,464.054028,1794.018758,178.651979,393.475469,470.670768,...,43.229368,715.049030,235.114826,775.040806,24.261380,1051.179420,1012.802328,187.474299,307.016734,162
B22_2,46.363146,593.237531,411.999777,362.475507,83.769776,404.623822,2161.681696,285.027979,385.130227,367.217193,...,77.974382,936.746297,190.194271,613.784834,23.708427,1295.006973,913.564724,285.027979,431.493373,49


In [117]:
brain_data_male = brain_sub_df_male.values

In [118]:
X, y = brain_data_male[:,:-1], brain_data_male[:,-1]

In [119]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.063e+00, tolerance: 5.057e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.551e+00, tolerance: 4.650e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.1, 'l1_ratio': 0.9}
Mean MAE: 17.394
Pearson correlation: 0.852
R-squared: 0.709


In [120]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 17.394
Pearson correlation: 0.852
R-squared: 0.709
Average number of non-zero coefficients: 107.48


#### Female

In [121]:
brain_df_female = brain_df.loc[brain_df.iloc[:,-3] == "F"]

brain_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A23_2,17.414194,584.876719,587.879166,315.256959,104.485163,422.744569,3009.653097,232.389415,351.286325,369.301009,...,210.771795,505.011623,31.225451,1231.003362,736.800549,254.007035,441.960231,F,Brain,78
C22_2,26.304342,542.317178,356.507784,341.956446,93.464364,463.404152,2857.099272,389.528128,366.022120,348.672448,...,314.532770,457.807484,7.835336,1242.460408,826.627938,146.073048,434.301476,F,Brain,103
C23_2,37.729860,592.298911,484.499311,353.343132,107.200713,560.557917,2921.968033,204.819239,523.426944,464.137164,...,219.192519,630.627657,49.108706,1183.998934,798.914810,276.086752,457.549411,F,Brain,49
D23_2,45.887753,606.536140,567.463400,387.092727,105.859866,459.786197,1926.831288,239.888451,454.788521,333.026959,...,175.827330,689.679296,36.346735,1182.177555,884.134328,265.785499,567.009066,F,Brain,75
E21_2,116.364825,539.509645,537.586259,388.043033,110.113822,399.102500,2589.357786,264.946358,389.004726,373.617642,...,195.223633,557.300961,47.122946,1332.425334,890.046660,320.243693,427.953283,F,Brain,102


In [122]:
brain_sub_df_female = brain_df_female.iloc[:,:-3]

brain_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A23_2,17.414194,584.876719,587.879166,315.256959,104.485163,422.744569,3009.653097,232.389415,351.286325,369.301009,...,2283.060869,65.453349,807.057814,210.771795,505.011623,31.225451,1231.003362,736.800549,254.007035,441.960231
C22_2,26.304342,542.317178,356.507784,341.956446,93.464364,463.404152,2857.099272,389.528128,366.022120,348.672448,...,2035.508335,65.481022,805.360598,314.532770,457.807484,7.835336,1242.460408,826.627938,146.073048,434.301476
C23_2,37.729860,592.298911,484.499311,353.343132,107.200713,560.557917,2921.968033,204.819239,523.426944,464.137164,...,1957.760504,75.459720,913.302162,219.192519,630.627657,49.108706,1183.998934,798.914810,276.086752,457.549411
D23_2,45.887753,606.536140,567.463400,387.092727,105.859866,459.786197,1926.831288,239.888451,454.788521,333.026959,...,2178.078093,61.789449,706.035327,175.827330,689.679296,36.346735,1182.177555,884.134328,265.785499,567.009066
E21_2,116.364825,539.509645,537.586259,388.043033,110.113822,399.102500,2589.357786,264.946358,389.004726,373.617642,...,1947.427862,61.548337,929.476063,195.223633,557.300961,47.122946,1332.425334,890.046660,320.243693,427.953283


In [123]:
brain_sub_df_female["age"] = brain_df_female["age_days"].tolist()

brain_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A23_2,17.414194,584.876719,587.879166,315.256959,104.485163,422.744569,3009.653097,232.389415,351.286325,369.301009,...,65.453349,807.057814,210.771795,505.011623,31.225451,1231.003362,736.800549,254.007035,441.960231,78
C22_2,26.304342,542.317178,356.507784,341.956446,93.464364,463.404152,2857.099272,389.528128,366.022120,348.672448,...,65.481022,805.360598,314.532770,457.807484,7.835336,1242.460408,826.627938,146.073048,434.301476,103
C23_2,37.729860,592.298911,484.499311,353.343132,107.200713,560.557917,2921.968033,204.819239,523.426944,464.137164,...,75.459720,913.302162,219.192519,630.627657,49.108706,1183.998934,798.914810,276.086752,457.549411,49
D23_2,45.887753,606.536140,567.463400,387.092727,105.859866,459.786197,1926.831288,239.888451,454.788521,333.026959,...,61.789449,706.035327,175.827330,689.679296,36.346735,1182.177555,884.134328,265.785499,567.009066,75
E21_2,116.364825,539.509645,537.586259,388.043033,110.113822,399.102500,2589.357786,264.946358,389.004726,373.617642,...,61.548337,929.476063,195.223633,557.300961,47.122946,1332.425334,890.046660,320.243693,427.953283,102


In [124]:
brain_data_female = brain_sub_df_female.values

In [125]:
X, y = brain_data_female[:,:-1], brain_data_female[:,-1]

In [126]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.687e+00, tolerance: 3.320e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.633e+00, tolerance: 3.293e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.001, 'l1_ratio': 0.9}
Mean MAE: 13.024
Pearson correlation: 0.923
R-squared: 0.840


In [127]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 13.024
Pearson correlation: 0.923
R-squared: 0.840
Average number of non-zero coefficients: 112.52


# Fat

In [128]:
fat_df = combined_df.loc[combined_df.iloc[:,-2] == "Fat"]

fat_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A5_2,8535.344945,282.528840,675.520083,437.601060,227.297638,624.537435,1512.485217,686.141468,633.034543,422.731121,...,93.468188,1270.317640,57.355479,390.866966,1659.060329,274.031732,397.239797,M,Fat,52
A6_2,2969.048330,518.818239,99.478423,353.531012,205.078596,481.322526,746.088176,278.539586,511.931271,371.131041,...,143.861105,522.644332,147.687198,311.443987,3021.848416,188.243786,208.904689,M,Fat,77
A7_2,3743.681369,269.376862,279.889130,266.748795,174.766452,482.250285,1165.547692,319.310134,448.085415,210.245356,...,84.098142,876.460328,60.445540,373.185507,1625.459408,223.385691,320.624168,F,Fat,78
A8_2,3371.080449,322.962776,247.855154,373.034524,105.150671,274.142821,786.126447,280.401790,207.797755,186.517262,...,61.337892,579.580486,71.352241,394.315017,1107.837429,173.999325,265.380266,M,Fat,103
B5_2,5975.375798,410.423324,424.965883,507.373716,305.393733,665.726022,1176.331417,664.110182,533.227153,365.179808,...,67.865274,1315.293645,43.627676,484.751958,1515.657787,302.162054,337.710530,F,Fat,134


In [129]:
fat_df_male = fat_df.loc[fat_df.iloc[:,-3] == "M"]

fat_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A5_2,8535.344945,282.528840,675.520083,437.601060,227.297638,624.537435,1512.485217,686.141468,633.034543,422.731121,...,93.468188,1270.317640,57.355479,390.866966,1659.060329,274.031732,397.239797,M,Fat,52
A6_2,2969.048330,518.818239,99.478423,353.531012,205.078596,481.322526,746.088176,278.539586,511.931271,371.131041,...,143.861105,522.644332,147.687198,311.443987,3021.848416,188.243786,208.904689,M,Fat,77
A8_2,3371.080449,322.962776,247.855154,373.034524,105.150671,274.142821,786.126447,280.401790,207.797755,186.517262,...,61.337892,579.580486,71.352241,394.315017,1107.837429,173.999325,265.380266,M,Fat,103
B7_2,2024.601159,607.466027,159.363443,397.551814,323.867642,497.796561,714.565115,582.619038,418.971632,533.781854,...,156.793065,702.570017,133.659662,365.850484,3573.682367,217.625347,282.741592,M,Fat,102
B8_2,2079.848143,300.129985,168.494027,235.628366,211.933893,469.940372,655.547073,594.994532,494.951204,242.210163,...,72.399777,685.823343,130.319599,321.191739,1396.657519,147.432273,201.403016,M,Fat,152


In [130]:
fat_sub_df_male = fat_df_male.iloc[:,:-3]
fat_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A5_2,8535.344945,282.528840,675.520083,437.601060,227.297638,624.537435,1512.485217,686.141468,633.034543,422.731121,...,301.647333,59.479756,996.285908,93.468188,1270.317640,57.355479,390.866966,1659.060329,274.031732,397.239797
A6_2,2969.048330,518.818239,99.478423,353.531012,205.078596,481.322526,746.088176,278.539586,511.931271,371.131041,...,407.861536,63.513147,863.931847,143.861105,522.644332,147.687198,311.443987,3021.848416,188.243786,208.904689
A8_2,3371.080449,322.962776,247.855154,373.034524,105.150671,274.142821,786.126447,280.401790,207.797755,186.517262,...,250.358741,75.107622,444.386765,61.337892,579.580486,71.352241,394.315017,1107.837429,173.999325,265.380266
B7_2,2024.601159,607.466027,159.363443,397.551814,323.867642,497.796561,714.565115,582.619038,418.971632,533.781854,...,394.981436,135.373247,1021.296903,156.793065,702.570017,133.659662,365.850484,3573.682367,217.625347,282.741592
B8_2,2079.848143,300.129985,168.494027,235.628366,211.933893,469.940372,655.547073,594.994532,494.951204,242.210163,...,188.239421,98.726969,513.380238,72.399777,685.823343,130.319599,321.191739,1396.657519,147.432273,201.403016


In [131]:
fat_sub_df_male["age"] = fat_df_male["age_days"].tolist()

fat_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A5_2,8535.344945,282.528840,675.520083,437.601060,227.297638,624.537435,1512.485217,686.141468,633.034543,422.731121,...,59.479756,996.285908,93.468188,1270.317640,57.355479,390.866966,1659.060329,274.031732,397.239797,52
A6_2,2969.048330,518.818239,99.478423,353.531012,205.078596,481.322526,746.088176,278.539586,511.931271,371.131041,...,63.513147,863.931847,143.861105,522.644332,147.687198,311.443987,3021.848416,188.243786,208.904689,77
A8_2,3371.080449,322.962776,247.855154,373.034524,105.150671,274.142821,786.126447,280.401790,207.797755,186.517262,...,75.107622,444.386765,61.337892,579.580486,71.352241,394.315017,1107.837429,173.999325,265.380266,103
B7_2,2024.601159,607.466027,159.363443,397.551814,323.867642,497.796561,714.565115,582.619038,418.971632,533.781854,...,135.373247,1021.296903,156.793065,702.570017,133.659662,365.850484,3573.682367,217.625347,282.741592,102
B8_2,2079.848143,300.129985,168.494027,235.628366,211.933893,469.940372,655.547073,594.994532,494.951204,242.210163,...,98.726969,513.380238,72.399777,685.823343,130.319599,321.191739,1396.657519,147.432273,201.403016,152


In [132]:
fat_data_male = fat_sub_df_male.values

In [133]:
X, y = fat_data_male[:,:-1], fat_data_male[:,-1]

In [134]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.962e+00, tolerance: 4.920e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.257e+00, tolerance: 5.160e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.001, 'l1_ratio': 0.0}
Mean MAE: 21.284
Pearson correlation: 0.805
R-squared: 0.646


In [135]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 21.284
Pearson correlation: 0.805
R-squared: 0.646
Average number of non-zero coefficients: 25121.07


##### Female

In [136]:
fat_df_female = fat_df.loc[fat_df.iloc[:,-3] == "F"]

fat_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A7_2,3743.681369,269.376862,279.889130,266.748795,174.766452,482.250285,1165.547692,319.310134,448.085415,210.245356,...,84.098142,876.460328,60.445540,373.185507,1625.459408,223.385691,320.624168,F,Fat,78
B5_2,5975.375798,410.423324,424.965883,507.373716,305.393733,665.726022,1176.331417,664.110182,533.227153,365.179808,...,67.865274,1315.293645,43.627676,484.751958,1515.657787,302.162054,337.710530,F,Fat,134
B6_2,4114.594864,390.359000,168.803892,346.047978,331.277638,531.732259,915.761113,818.698875,453.660459,253.205838,...,46.421070,808.148632,71.741654,362.928367,2160.689816,238.435497,373.478611,F,Fat,133
C8_2,2433.754524,906.092062,170.712997,472.743685,490.252710,1212.500006,485.875454,1426.985566,792.283397,437.725634,...,48.149820,976.128163,91.922383,240.749099,1343.817696,56.904332,100.676896,F,Fat,47
D5_2,2991.517298,426.536185,123.926054,329.989143,232.001101,531.729231,858.836373,632.599274,479.853208,286.759124,...,70.609031,835.780363,95.106041,305.492133,2180.233946,198.858086,272.349118,F,Fat,102


In [137]:
fat_sub_df_female = fat_df_female.iloc[:,:-3]

fat_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A7_2,3743.681369,269.376862,279.889130,266.748795,174.766452,482.250285,1165.547692,319.310134,448.085415,210.245356,...,240.468126,63.073607,982.897039,84.098142,876.460328,60.445540,373.185507,1625.459408,223.385691,320.624168
B5_2,5975.375798,410.423324,424.965883,507.373716,305.393733,665.726022,1176.331417,664.110182,533.227153,365.179808,...,321.552132,105.029591,1095.539424,67.865274,1315.293645,43.627676,484.751958,1515.657787,302.162054,337.710530
B6_2,4114.594864,390.359000,168.803892,346.047978,331.277638,531.732259,915.761113,818.698875,453.660459,253.205838,...,291.186713,103.392384,763.837611,46.421070,808.148632,71.741654,362.928367,2160.689816,238.435497,373.478611
C8_2,2433.754524,906.092062,170.712997,472.743685,490.252710,1212.500006,485.875454,1426.985566,792.283397,437.725634,...,48.149820,218.862817,284.521662,48.149820,976.128163,91.922383,240.749099,1343.817696,56.904332,100.676896
D5_2,2991.517298,426.536185,123.926054,329.989143,232.001101,531.729231,858.836373,632.599274,479.853208,286.759124,...,327.107142,102.311044,642.686279,70.609031,835.780363,95.106041,305.492133,2180.233946,198.858086,272.349118


In [138]:
fat_sub_df_female["age"] = fat_df_female["age_days"].tolist()

fat_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A7_2,3743.681369,269.376862,279.889130,266.748795,174.766452,482.250285,1165.547692,319.310134,448.085415,210.245356,...,63.073607,982.897039,84.098142,876.460328,60.445540,373.185507,1625.459408,223.385691,320.624168,78
B5_2,5975.375798,410.423324,424.965883,507.373716,305.393733,665.726022,1176.331417,664.110182,533.227153,365.179808,...,105.029591,1095.539424,67.865274,1315.293645,43.627676,484.751958,1515.657787,302.162054,337.710530,134
B6_2,4114.594864,390.359000,168.803892,346.047978,331.277638,531.732259,915.761113,818.698875,453.660459,253.205838,...,103.392384,763.837611,46.421070,808.148632,71.741654,362.928367,2160.689816,238.435497,373.478611,133
C8_2,2433.754524,906.092062,170.712997,472.743685,490.252710,1212.500006,485.875454,1426.985566,792.283397,437.725634,...,218.862817,284.521662,48.149820,976.128163,91.922383,240.749099,1343.817696,56.904332,100.676896,47
D5_2,2991.517298,426.536185,123.926054,329.989143,232.001101,531.729231,858.836373,632.599274,479.853208,286.759124,...,102.311044,642.686279,70.609031,835.780363,95.106041,305.492133,2180.233946,198.858086,272.349118,102


In [139]:
fat_data_female = fat_sub_df_female.values

In [140]:
X, y = fat_data_female[:,:-1], fat_data_female[:,-1]

In [141]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.608e+00, tolerance: 3.321e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.350e+00, tolerance: 3.077e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.01, 'l1_ratio': 0.0}
Mean MAE: 21.328
Pearson correlation: 0.736
R-squared: 0.536


In [142]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 21.328
Pearson correlation: 0.736
R-squared: 0.536
Average number of non-zero coefficients: 25122.00


# Bone

In [143]:
bone_df = combined_df.loc[combined_df.iloc[:,-2] == "Bone"]

bone_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A1_2,36.845369,407.384647,349.683408,374.015256,493.588906,469.257059,963.541161,1619.805848,453.962755,449.791581,...,107.060129,777.923924,19.465478,672.254187,2691.797529,191.873997,253.746410,M,Bone,78
A2_2,20.777004,442.878244,147.626081,246.043469,347.741436,461.468195,1131.799958,2439.657687,538.015052,407.885395,...,86.388596,600.346065,18.589951,461.468195,2926.276993,346.647910,331.338538,M,Bone,133
A3_2,2689.339698,509.924698,228.124207,424.674129,254.172992,471.246199,1022.217467,1155.618819,357.578774,475.192984,...,87.618640,858.031186,67.884712,356.000060,1692.381659,211.547707,330.740632,F,Bone,102
A4_2,22.871884,461.415405,139.220165,452.465537,327.167388,432.576942,622.513025,1147.571934,618.535306,353.022562,...,175.019636,540.969785,50.715917,401.749620,2937.545488,133.253587,179.991785,M,Bone,77
B1_2,2192.314285,358.300673,188.259675,432.389964,368.017301,556.276976,804.051001,1699.195393,572.066498,364.373565,...,25.506150,517.410463,104.453755,593.928912,2108.508365,102.024598,224.697032,M,Bone,134


In [144]:
bone_df_male = bone_df.loc[bone_df.iloc[:,-3] == "M"]

bone_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A1_2,36.845369,407.384647,349.683408,374.015256,493.588906,469.257059,963.541161,1619.805848,453.962755,449.791581,...,107.060129,777.923924,19.465478,672.254187,2691.797529,191.873997,253.746410,M,Bone,78
A2_2,20.777004,442.878244,147.626081,246.043469,347.741436,461.468195,1131.799958,2439.657687,538.015052,407.885395,...,86.388596,600.346065,18.589951,461.468195,2926.276993,346.647910,331.338538,M,Bone,133
A4_2,22.871884,461.415405,139.220165,452.465537,327.167388,432.576942,622.513025,1147.571934,618.535306,353.022562,...,175.019636,540.969785,50.715917,401.749620,2937.545488,133.253587,179.991785,M,Bone,77
B1_2,2192.314285,358.300673,188.259675,432.389964,368.017301,556.276976,804.051001,1699.195393,572.066498,364.373565,...,25.506150,517.410463,104.453755,593.928912,2108.508365,102.024598,224.697032,M,Bone,134
B2_2,763.395427,433.326190,161.504466,608.068727,387.434211,537.465682,879.890451,1461.483036,422.735733,447.446799,...,73.250659,702.500300,74.133197,679.554310,3687.244037,300.062942,347.719998,M,Bone,152


In [145]:
bone_sub_df_male = bone_df_male.iloc[:,:-3]
bone_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A1_2,36.845369,407.384647,349.683408,374.015256,493.588906,469.257059,963.541161,1619.805848,453.962755,449.791581,...,774.447946,64.653195,670.168600,107.060129,777.923924,19.465478,672.254187,2691.797529,191.873997,253.746410
A2_2,20.777004,442.878244,147.626081,246.043469,347.741436,461.468195,1131.799958,2439.657687,538.015052,407.885395,...,898.878807,73.266277,598.159012,86.388596,600.346065,18.589951,461.468195,2926.276993,346.647910,331.338538
A4_2,22.871884,461.415405,139.220165,452.465537,327.167388,432.576942,622.513025,1147.571934,618.535306,353.022562,...,435.560231,50.715917,875.098182,175.019636,540.969785,50.715917,401.749620,2937.545488,133.253587,179.991785
B1_2,2192.314285,358.300673,188.259675,432.389964,368.017301,556.276976,804.051001,1699.195393,572.066498,364.373565,...,702.026403,111.741227,947.371270,25.506150,517.410463,104.453755,593.928912,2108.508365,102.024598,224.697032
B2_2,763.395427,433.326190,161.504466,608.068727,387.434211,537.465682,879.890451,1461.483036,422.735733,447.446799,...,725.446290,70.603045,861.357152,73.250659,702.500300,74.133197,679.554310,3687.244037,300.062942,347.719998


In [146]:
bone_sub_df_male["age"] = bone_df_male["age_days"].tolist()

bone_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A1_2,36.845369,407.384647,349.683408,374.015256,493.588906,469.257059,963.541161,1619.805848,453.962755,449.791581,...,64.653195,670.168600,107.060129,777.923924,19.465478,672.254187,2691.797529,191.873997,253.746410,78
A2_2,20.777004,442.878244,147.626081,246.043469,347.741436,461.468195,1131.799958,2439.657687,538.015052,407.885395,...,73.266277,598.159012,86.388596,600.346065,18.589951,461.468195,2926.276993,346.647910,331.338538,133
A4_2,22.871884,461.415405,139.220165,452.465537,327.167388,432.576942,622.513025,1147.571934,618.535306,353.022562,...,50.715917,875.098182,175.019636,540.969785,50.715917,401.749620,2937.545488,133.253587,179.991785,77
B1_2,2192.314285,358.300673,188.259675,432.389964,368.017301,556.276976,804.051001,1699.195393,572.066498,364.373565,...,111.741227,947.371270,25.506150,517.410463,104.453755,593.928912,2108.508365,102.024598,224.697032,134
B2_2,763.395427,433.326190,161.504466,608.068727,387.434211,537.465682,879.890451,1461.483036,422.735733,447.446799,...,70.603045,861.357152,73.250659,702.500300,74.133197,679.554310,3687.244037,300.062942,347.719998,152


In [147]:
bone_data_male = bone_sub_df_male.values

In [148]:
X, y = bone_data_male[:,:-1], bone_data_male[:,-1]

In [149]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.446e+00, tolerance: 5.337e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.482e+00, tolerance: 5.342e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 1.0, 'l1_ratio': 1.0}
Mean MAE: 103.854
Pearson correlation: 0.323
R-squared: -100.087


In [150]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 103.854
Pearson correlation: 0.323
R-squared: -100.087
Average number of non-zero coefficients: 26.43


##### Female

In [151]:
bone_df_female = bone_df.loc[bone_df.iloc[:,-3] == "F"]

bone_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A3_2,2689.339698,509.924698,228.124207,424.674129,254.172992,471.246199,1022.217467,1155.618819,357.578774,475.192984,...,87.618640,858.031186,67.884712,356.000060,1692.381659,211.547707,330.740632,F,Bone,102
C3_2,114.608374,593.671377,289.386144,424.624025,536.367190,503.130761,946.092126,1316.277174,558.142781,547.254985,...,73.349359,684.785034,78.506736,497.973385,2954.030837,259.587967,268.183595,F,Bone,155
C4_2,298.910593,455.393567,362.628211,336.391545,312.965950,351.383926,1048.529634,582.828804,364.502259,380.431663,...,92.765356,596.884162,53.410357,720.571303,1555.459510,277.359045,288.603331,F,Bone,49
D2_2,26.262471,418.388329,321.488868,354.090556,538.833454,668.334604,888.395998,1410.023006,614.904060,465.479656,...,83.315425,727.198763,65.203376,482.686103,1790.376033,259.907902,335.978507,F,Bone,75
E2_2,739.966763,739.357737,236.911169,275.888843,460.423764,504.882672,723.523057,547.514502,637.041345,568.830417,...,54.203327,964.088383,152.256536,285.024235,2182.140668,274.061764,244.219483,F,Bone,47


In [152]:
bone_sub_df_female = bone_df_female.iloc[:,:-3]

bone_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A3_2,2689.339698,509.924698,228.124207,424.674129,254.172992,471.246199,1022.217467,1155.618819,357.578774,475.192984,...,419.148629,131.822639,958.279540,87.618640,858.031186,67.884712,356.000060,1692.381659,211.547707,330.740632
C3_2,114.608374,593.671377,289.386144,424.624025,536.367190,503.130761,946.092126,1316.277174,558.142781,547.254985,...,741.516179,134.091797,700.257164,73.349359,684.785034,78.506736,497.973385,2954.030837,259.587967,268.183595
C4_2,298.910593,455.393567,362.628211,336.391545,312.965950,351.383926,1048.529634,582.828804,364.502259,380.431663,...,1271.541299,67.465714,963.260468,92.765356,596.884162,53.410357,720.571303,1555.459510,277.359045,288.603331
D2_2,26.262471,418.388329,321.488868,354.090556,538.833454,668.334604,888.395998,1410.023006,614.904060,465.479656,...,870.283949,102.333076,683.729846,83.315425,727.198763,65.203376,482.686103,1790.376033,259.907902,335.978507
E2_2,739.966763,739.357737,236.911169,275.888843,460.423764,504.882672,723.523057,547.514502,637.041345,568.830417,...,827.666528,123.632307,617.552508,54.203327,964.088383,152.256536,285.024235,2182.140668,274.061764,244.219483


In [153]:
bone_sub_df_female["age"] = bone_df_female["age_days"].tolist()

bone_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A3_2,2689.339698,509.924698,228.124207,424.674129,254.172992,471.246199,1022.217467,1155.618819,357.578774,475.192984,...,131.822639,958.279540,87.618640,858.031186,67.884712,356.000060,1692.381659,211.547707,330.740632,102
C3_2,114.608374,593.671377,289.386144,424.624025,536.367190,503.130761,946.092126,1316.277174,558.142781,547.254985,...,134.091797,700.257164,73.349359,684.785034,78.506736,497.973385,2954.030837,259.587967,268.183595,155
C4_2,298.910593,455.393567,362.628211,336.391545,312.965950,351.383926,1048.529634,582.828804,364.502259,380.431663,...,67.465714,963.260468,92.765356,596.884162,53.410357,720.571303,1555.459510,277.359045,288.603331,49
D2_2,26.262471,418.388329,321.488868,354.090556,538.833454,668.334604,888.395998,1410.023006,614.904060,465.479656,...,102.333076,683.729846,83.315425,727.198763,65.203376,482.686103,1790.376033,259.907902,335.978507,75
E2_2,739.966763,739.357737,236.911169,275.888843,460.423764,504.882672,723.523057,547.514502,637.041345,568.830417,...,123.632307,617.552508,54.203327,964.088383,152.256536,285.024235,2182.140668,274.061764,244.219483,47


In [154]:
bone_data_female = bone_sub_df_female.values

In [155]:
X, y = bone_data_female[:,:-1], bone_data_female[:,-1]

In [156]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.929e+00, tolerance: 2.812e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.166e+00, tolerance: 2.987e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.01, 'l1_ratio': 1.0}
Mean MAE: 13.982
Pearson correlation: 0.900
R-squared: 0.772


In [157]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 13.982
Pearson correlation: 0.900
R-squared: 0.772
Average number of non-zero coefficients: 37.62


# Eye

In [158]:
eye_df = combined_df.loc[combined_df.iloc[:,-2] == "Eye"]

eye_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
I12_2,13.005389,463.464767,712.931771,812.245650,95.766954,668.004064,3055.084074,1345.466593,1112.551901,611.253276,...,242.373156,509.574782,31.922318,1062.894962,786.234872,200.992373,516.668630,M,Eye,47
I16_2,5.068039,355.776306,680.130774,867.648200,92.238302,642.627288,2904.999697,1141.322282,948.736817,580.797218,...,337.531367,531.130440,49.666778,1061.247273,1059.220057,188.531034,490.586132,M,Eye,147
I20_2,20.535311,531.502170,858.859189,1075.083935,160.658611,1063.004341,2765.019245,1606.586106,1120.986396,950.664109,...,378.091317,682.497105,22.951230,822.620405,718.735889,80.933285,397.418668,M,Eye,78
I4_2,15.386990,386.484981,886.109592,859.861198,109.519163,629.961467,3569.781647,1411.982599,1011.920862,604.618190,...,244.381604,510.486016,26.248394,943.131967,1116.914440,115.854982,492.383675,F,Eye,103
I8_2,12.334831,532.640416,1043.975216,1095.557235,133.440441,821.948264,3856.316615,1303.006661,1168.444871,788.307816,...,316.220205,512.456148,43.732582,990.150500,729.997708,124.469655,420.505592,F,Eye,78


In [159]:
eye_df_male = eye_df.loc[eye_df.iloc[:,-3] == "M"]

eye_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
I12_2,13.005389,463.464767,712.931771,812.245650,95.766954,668.004064,3055.084074,1345.466593,1112.551901,611.253276,...,242.373156,509.574782,31.922318,1062.894962,786.234872,200.992373,516.668630,M,Eye,47
I16_2,5.068039,355.776306,680.130774,867.648200,92.238302,642.627288,2904.999697,1141.322282,948.736817,580.797218,...,337.531367,531.130440,49.666778,1061.247273,1059.220057,188.531034,490.586132,M,Eye,147
I20_2,20.535311,531.502170,858.859189,1075.083935,160.658611,1063.004341,2765.019245,1606.586106,1120.986396,950.664109,...,378.091317,682.497105,22.951230,822.620405,718.735889,80.933285,397.418668,M,Eye,78
J12_2,19.377526,439.563875,617.021217,950.518635,96.887629,742.465200,1914.295577,606.822519,705.749888,485.458015,...,311.060283,666.994836,27.536484,1098.399753,925.021891,137.682420,661.895487,M,Eye,103
J20_2,13.585083,473.666574,770.727063,820.539036,90.567222,678.348496,3365.477987,1110.354147,862.199958,532.535268,...,252.682551,605.894718,8.151050,812.387986,843.180841,192.908184,518.950185,M,Eye,147


In [160]:
eye_sub_df_male = eye_df_male.iloc[:,:-3]
eye_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
I12_2,13.005389,463.464767,712.931771,812.245650,95.766954,668.004064,3055.084074,1345.466593,1112.551901,611.253276,...,1033.337260,295.577020,1254.428871,242.373156,509.574782,31.922318,1062.894962,786.234872,200.992373,516.668630
I16_2,5.068039,355.776306,680.130774,867.648200,92.238302,642.627288,2904.999697,1141.322282,948.736817,580.797218,...,1093.682719,266.578828,969.008971,337.531367,531.130440,49.666778,1061.247273,1059.220057,188.531034,490.586132
I20_2,20.535311,531.502170,858.859189,1075.083935,160.658611,1063.004341,2765.019245,1606.586106,1120.986396,950.664109,...,623.307091,353.932127,756.182633,378.091317,682.497105,22.951230,822.620405,718.735889,80.933285,397.418668
J12_2,19.377526,439.563875,617.021217,950.518635,96.887629,742.465200,1914.295577,606.822519,705.749888,485.458015,...,1073.922879,139.722160,1436.996520,311.060283,666.994836,27.536484,1098.399753,925.021891,137.682420,661.895487
J20_2,13.585083,473.666574,770.727063,820.539036,90.567222,678.348496,3365.477987,1110.354147,862.199958,532.535268,...,1120.316542,222.795367,1256.167376,252.682551,605.894718,8.151050,812.387986,843.180841,192.908184,518.950185


In [161]:
eye_sub_df_male["age"] = eye_df_male["age_days"].tolist()

eye_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
I12_2,13.005389,463.464767,712.931771,812.245650,95.766954,668.004064,3055.084074,1345.466593,1112.551901,611.253276,...,295.577020,1254.428871,242.373156,509.574782,31.922318,1062.894962,786.234872,200.992373,516.668630,47
I16_2,5.068039,355.776306,680.130774,867.648200,92.238302,642.627288,2904.999697,1141.322282,948.736817,580.797218,...,266.578828,969.008971,337.531367,531.130440,49.666778,1061.247273,1059.220057,188.531034,490.586132,147
I20_2,20.535311,531.502170,858.859189,1075.083935,160.658611,1063.004341,2765.019245,1606.586106,1120.986396,950.664109,...,353.932127,756.182633,378.091317,682.497105,22.951230,822.620405,718.735889,80.933285,397.418668,78
J12_2,19.377526,439.563875,617.021217,950.518635,96.887629,742.465200,1914.295577,606.822519,705.749888,485.458015,...,139.722160,1436.996520,311.060283,666.994836,27.536484,1098.399753,925.021891,137.682420,661.895487,103
J20_2,13.585083,473.666574,770.727063,820.539036,90.567222,678.348496,3365.477987,1110.354147,862.199958,532.535268,...,222.795367,1256.167376,252.682551,605.894718,8.151050,812.387986,843.180841,192.908184,518.950185,147


In [162]:
eye_data_male = eye_sub_df_male.values

In [163]:
X, y = eye_data_male[:,:-1], eye_data_male[:,-1]

In [164]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.512e+00, tolerance: 3.237e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.867e+00, tolerance: 2.764e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 1.0, 'l1_ratio': 1.0}
Mean MAE: 16.350
Pearson correlation: 0.856
R-squared: 0.717


In [165]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 16.350
Pearson correlation: 0.856
R-squared: 0.717
Average number of non-zero coefficients: 19.57


##### Female

In [166]:
eye_df_female = eye_df.loc[eye_df.iloc[:,-3] == "F"]

eye_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
I4_2,15.386990,386.484981,886.109592,859.861198,109.519163,629.961467,3569.781647,1411.982599,1011.920862,604.618190,...,244.381604,510.486016,26.248394,943.131967,1116.914440,115.854982,492.383675,F,Eye,103
I8_2,12.334831,532.640416,1043.975216,1095.557235,133.440441,821.948264,3856.316615,1303.006661,1168.444871,788.307816,...,316.220205,512.456148,43.732582,990.150500,729.997708,124.469655,420.505592,F,Eye,78
J16_2,22.685047,429.201097,754.050975,980.901449,107.980826,695.069852,3067.925808,1020.827132,767.662004,589.811232,...,382.923600,676.014412,6.351813,1063.475021,842.068959,164.239743,499.978444,F,Eye,147
J4_2,12.046830,512.659560,938.314234,1137.756203,132.515134,821.861540,2374.564123,2248.741672,1127.047910,700.054699,...,319.910274,661.237135,16.062441,919.574720,527.383464,95.036106,285.108319,F,Eye,75
K4_2,15.665415,519.432180,642.282013,1146.873274,50.294227,738.747989,1886.445759,812.128091,779.972766,577.971362,...,357.831057,676.086330,4.122478,937.451411,878.087733,157.478645,583.742831,F,Eye,103


In [167]:
eye_sub_df_female = eye_df_female.iloc[:,:-3]

eye_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
I4_2,15.386990,386.484981,886.109592,859.861198,109.519163,629.961467,3569.781647,1411.982599,1011.920862,604.618190,...,1175.747049,297.783510,912.357987,244.381604,510.486016,26.248394,943.131967,1116.914440,115.854982,492.383675
I8_2,12.334831,532.640416,1043.975216,1095.557235,133.440441,821.948264,3856.316615,1303.006661,1168.444871,788.307816,...,1066.402181,305.006723,1040.611171,316.220205,512.456148,43.732582,990.150500,729.997708,124.469655,420.505592
J16_2,22.685047,429.201097,754.050975,980.901449,107.980826,695.069852,3067.925808,1020.827132,767.662004,589.811232,...,1107.937714,241.368904,1186.881679,382.923600,676.014412,6.351813,1063.475021,842.068959,164.239743,499.978444
J4_2,12.046830,512.659560,938.314234,1137.756203,132.515134,821.861540,2374.564123,2248.741672,1127.047910,700.054699,...,915.559109,489.904436,1168.542548,319.910274,661.237135,16.062441,919.574720,527.383464,95.036106,285.108319
K4_2,15.665415,519.432180,642.282013,1146.873274,50.294227,738.747989,1886.445759,812.128091,779.972766,577.971362,...,1025.672432,157.478645,1292.808981,357.831057,676.086330,4.122478,937.451411,878.087733,157.478645,583.742831


In [168]:
eye_sub_df_female["age"] = eye_df_female["age_days"].tolist()

eye_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
I4_2,15.386990,386.484981,886.109592,859.861198,109.519163,629.961467,3569.781647,1411.982599,1011.920862,604.618190,...,297.783510,912.357987,244.381604,510.486016,26.248394,943.131967,1116.914440,115.854982,492.383675,103
I8_2,12.334831,532.640416,1043.975216,1095.557235,133.440441,821.948264,3856.316615,1303.006661,1168.444871,788.307816,...,305.006723,1040.611171,316.220205,512.456148,43.732582,990.150500,729.997708,124.469655,420.505592,78
J16_2,22.685047,429.201097,754.050975,980.901449,107.980826,695.069852,3067.925808,1020.827132,767.662004,589.811232,...,241.368904,1186.881679,382.923600,676.014412,6.351813,1063.475021,842.068959,164.239743,499.978444,147
J4_2,12.046830,512.659560,938.314234,1137.756203,132.515134,821.861540,2374.564123,2248.741672,1127.047910,700.054699,...,489.904436,1168.542548,319.910274,661.237135,16.062441,919.574720,527.383464,95.036106,285.108319,75
K4_2,15.665415,519.432180,642.282013,1146.873274,50.294227,738.747989,1886.445759,812.128091,779.972766,577.971362,...,157.478645,1292.808981,357.831057,676.086330,4.122478,937.451411,878.087733,157.478645,583.742831,103


In [169]:
eye_data_female = eye_sub_df_female.values

In [170]:
X, y = eye_data_female[:,:-1], eye_data_female[:,-1]

In [171]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.573e+00, tolerance: 1.139e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.574e+00, tolerance: 1.138e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.0001, 'l1_ratio': 0.30000000000000004}
Mean MAE: 18.037
Pearson correlation: 0.578
R-squared: 0.159


In [172]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 18.037
Pearson correlation: 0.578
R-squared: 0.159
Average number of non-zero coefficients: 380.64


# Testis

In [173]:
testis_df = combined_df.loc[combined_df.iloc[:,-2] == "Testis"]

testis_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A21_1,22.364947,1328.342293,113.180185,474.407962,100.981123,1706.513211,481.862944,913.574189,1114.858710,1381.882620,...,3120.248938,2304.944969,412.057201,759.052739,1497.773708,495.417457,741.431872,M,Testis,78
A24_1,19.633246,1284.886904,170.154802,393.392086,67.625627,1447.042976,548.276585,820.233406,1060.922464,1291.431320,...,2309.451504,2417.070781,380.303255,841.320967,1469.584852,382.484726,591.906021,M,Testis,75
B22_1,13.034212,1248.217471,158.710698,428.595557,89.706047,1576.372925,536.702844,748.317108,1272.752459,1311.088376,...,3242.451896,2425.896855,380.292301,759.051165,1414.595353,545.903464,708.447754,M,Testis,133
B23_1,25.862520,1336.230182,126.726346,576.734188,143.105942,1785.375940,450.869926,936.223211,1348.299358,1558.647851,...,3352.644631,2305.212585,460.352850,765.530582,1347.437274,448.283674,656.045915,M,Testis,162
C21_1,4.611588,1344.738941,131.891405,432.566916,96.843339,1724.733759,571.836861,865.133832,1239.594744,1393.621770,...,2798.311350,2372.200657,471.304252,869.745420,1384.398595,500.818412,736.009379,M,Testis,134


In [174]:
testis_df_male = testis_df.loc[testis_df.iloc[:,-3] == "M"]

testis_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A21_1,22.364947,1328.342293,113.180185,474.407962,100.981123,1706.513211,481.862944,913.574189,1114.858710,1381.882620,...,3120.248938,2304.944969,412.057201,759.052739,1497.773708,495.417457,741.431872,M,Testis,78
A24_1,19.633246,1284.886904,170.154802,393.392086,67.625627,1447.042976,548.276585,820.233406,1060.922464,1291.431320,...,2309.451504,2417.070781,380.303255,841.320967,1469.584852,382.484726,591.906021,M,Testis,75
B22_1,13.034212,1248.217471,158.710698,428.595557,89.706047,1576.372925,536.702844,748.317108,1272.752459,1311.088376,...,3242.451896,2425.896855,380.292301,759.051165,1414.595353,545.903464,708.447754,M,Testis,133
B23_1,25.862520,1336.230182,126.726346,576.734188,143.105942,1785.375940,450.869926,936.223211,1348.299358,1558.647851,...,3352.644631,2305.212585,460.352850,765.530582,1347.437274,448.283674,656.045915,M,Testis,162
C21_1,4.611588,1344.738941,131.891405,432.566916,96.843339,1724.733759,571.836861,865.133832,1239.594744,1393.621770,...,2798.311350,2372.200657,471.304252,869.745420,1384.398595,500.818412,736.009379,M,Testis,134


In [175]:
testis_sub_df_male = testis_df_male.iloc[:,:-3]
testis_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A21_1,22.364947,1328.342293,113.180185,474.407962,100.981123,1706.513211,481.862944,913.574189,1114.858710,1381.882620,...,193.829539,797.005376,2551.637109,3120.248938,2304.944969,412.057201,759.052739,1497.773708,495.417457,741.431872
A24_1,19.633246,1284.886904,170.154802,393.392086,67.625627,1447.042976,548.276585,820.233406,1060.922464,1291.431320,...,230.508856,541.005012,2004.045449,2309.451504,2417.070781,380.303255,841.320967,1469.584852,382.484726,591.906021
B22_1,13.034212,1248.217471,158.710698,428.595557,89.706047,1576.372925,536.702844,748.317108,1272.752459,1311.088376,...,255.317210,670.878555,2646.711739,3242.451896,2425.896855,380.292301,759.051165,1414.595353,545.903464,708.447754
B23_1,25.862520,1336.230182,126.726346,576.734188,143.105942,1785.375940,450.869926,936.223211,1348.299358,1558.647851,...,235.348929,856.049400,2102.622848,3352.644631,2305.212585,460.352850,765.530582,1347.437274,448.283674,656.045915
C21_1,4.611588,1344.738941,131.891405,432.566916,96.843339,1724.733759,571.836861,865.133832,1239.594744,1393.621770,...,240.724872,749.844142,2418.316532,2798.311350,2372.200657,471.304252,869.745420,1384.398595,500.818412,736.009379


In [176]:
testis_sub_df_male["age"] = testis_df_male["age_days"].tolist()

testis_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A21_1,22.364947,1328.342293,113.180185,474.407962,100.981123,1706.513211,481.862944,913.574189,1114.858710,1381.882620,...,797.005376,2551.637109,3120.248938,2304.944969,412.057201,759.052739,1497.773708,495.417457,741.431872,78
A24_1,19.633246,1284.886904,170.154802,393.392086,67.625627,1447.042976,548.276585,820.233406,1060.922464,1291.431320,...,541.005012,2004.045449,2309.451504,2417.070781,380.303255,841.320967,1469.584852,382.484726,591.906021,75
B22_1,13.034212,1248.217471,158.710698,428.595557,89.706047,1576.372925,536.702844,748.317108,1272.752459,1311.088376,...,670.878555,2646.711739,3242.451896,2425.896855,380.292301,759.051165,1414.595353,545.903464,708.447754,133
B23_1,25.862520,1336.230182,126.726346,576.734188,143.105942,1785.375940,450.869926,936.223211,1348.299358,1558.647851,...,856.049400,2102.622848,3352.644631,2305.212585,460.352850,765.530582,1347.437274,448.283674,656.045915,162
C21_1,4.611588,1344.738941,131.891405,432.566916,96.843339,1724.733759,571.836861,865.133832,1239.594744,1393.621770,...,749.844142,2418.316532,2798.311350,2372.200657,471.304252,869.745420,1384.398595,500.818412,736.009379,134


In [177]:
testis_data_male = testis_sub_df_male.values

In [178]:
X, y = testis_data_male[:,:-1], testis_data_male[:,-1]

In [179]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.465e+00, tolerance: 5.338e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.421e+00, tolerance: 5.312e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.1, 'l1_ratio': 1.0}
Mean MAE: 7.938
Pearson correlation: 0.983
R-squared: 0.949


In [180]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 7.938
Pearson correlation: 0.983
R-squared: 0.949
Average number of non-zero coefficients: 31.81


# Ovary

In [181]:
ovary_df = combined_df.loc[combined_df.iloc[:,-2] == "Ovary"]

ovary_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A23_1,27.012859,1133.939803,453.215750,226.307732,517.446326,841.600637,1213.177523,267.127164,1103.925515,486.831752,...,203.496873,1713.815848,447.212892,564.868901,4106.554892,321.753168,1279.208957,F,Ovary,78
B21_1,20.475689,1021.327380,437.360722,222.775499,488.959459,787.085495,1417.736724,405.418647,1060.640703,378.390737,...,171.176762,1375.147290,443.093915,438.179750,4603.753971,484.864321,1013.137104,F,Ovary,155
B24_1,39.057682,902.808709,557.692472,232.425221,511.591602,533.361457,1333.723788,291.331888,915.614506,449.483485,...,170.317104,1703.171040,416.188412,565.375950,4901.418913,508.390152,1235.119149,F,Ovary,75
C23_1,26.677866,1058.222034,417.953240,215.848192,462.416351,880.369592,1059.838875,370.256449,1092.175682,515.772084,...,229.591335,1485.876317,486.668957,425.229022,3709.031851,392.892214,1054.988353,F,Ovary,147
D21_1,48.765093,940.469648,580.536820,212.089452,515.516696,990.008790,1332.912538,223.700188,1429.668675,535.641972,...,252.340004,1655.691010,517.838843,601.436145,4955.462295,421.082707,1150.236952,F,Ovary,78


In [182]:
ovary_df.shape

(15, 25125)

In [183]:
ovary_sub_df = ovary_df.iloc[:,:-3]

ovary_sub_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A23_1,27.012859,1133.939803,453.215750,226.307732,517.446326,841.600637,1213.177523,267.127164,1103.925515,486.831752,...,453.816035,106.850865,1315.826388,203.496873,1713.815848,447.212892,564.868901,4106.554892,321.753168,1279.208957
B21_1,20.475689,1021.327380,437.360722,222.775499,488.959459,787.085495,1417.736724,405.418647,1060.640703,378.390737,...,588.880823,102.378446,1275.225927,171.176762,1375.147290,443.093915,438.179750,4603.753971,484.864321,1013.137104
B24_1,39.057682,902.808709,557.692472,232.425221,511.591602,533.361457,1333.723788,291.331888,915.614506,449.483485,...,740.815373,68.511016,1657.070169,170.317104,1703.171040,416.188412,565.375950,4901.418913,508.390152,1235.119149
C23_1,26.677866,1058.222034,417.953240,215.848192,462.416351,880.369592,1059.838875,370.256449,1092.175682,515.772084,...,539.216270,159.258778,1205.354510,229.591335,1485.876317,486.668957,425.229022,3709.031851,392.892214,1054.988353
D21_1,48.765093,940.469648,580.536820,212.089452,515.516696,990.008790,1332.912538,223.700188,1429.668675,535.641972,...,660.263876,127.718100,1145.592658,252.340004,1655.691010,517.838843,601.436145,4955.462295,421.082707,1150.236952


In [184]:
ovary_sub_df["age"] = ovary_df["age_days"].tolist()

ovary_sub_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A23_1,27.012859,1133.939803,453.215750,226.307732,517.446326,841.600637,1213.177523,267.127164,1103.925515,486.831752,...,106.850865,1315.826388,203.496873,1713.815848,447.212892,564.868901,4106.554892,321.753168,1279.208957,78
B21_1,20.475689,1021.327380,437.360722,222.775499,488.959459,787.085495,1417.736724,405.418647,1060.640703,378.390737,...,102.378446,1275.225927,171.176762,1375.147290,443.093915,438.179750,4603.753971,484.864321,1013.137104,155
B24_1,39.057682,902.808709,557.692472,232.425221,511.591602,533.361457,1333.723788,291.331888,915.614506,449.483485,...,68.511016,1657.070169,170.317104,1703.171040,416.188412,565.375950,4901.418913,508.390152,1235.119149,75
C23_1,26.677866,1058.222034,417.953240,215.848192,462.416351,880.369592,1059.838875,370.256449,1092.175682,515.772084,...,159.258778,1205.354510,229.591335,1485.876317,486.668957,425.229022,3709.031851,392.892214,1054.988353,147
D21_1,48.765093,940.469648,580.536820,212.089452,515.516696,990.008790,1332.912538,223.700188,1429.668675,535.641972,...,127.718100,1145.592658,252.340004,1655.691010,517.838843,601.436145,4955.462295,421.082707,1150.236952,78


In [185]:
ovary_data = ovary_sub_df.values

In [186]:
X, y = ovary_data[:,:-1], ovary_data[:,-1]

y

array([ 78., 155.,  75., 147.,  78.,  49.,  77.,  52., 155.,  75.,  49.,
       102., 134.,  52., 102.])

In [187]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.561e+00, tolerance: 2.071e+00
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.864e+00, tolerance: 2.081e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.1

Best parameters found:  {'alpha': 0.01, 'l1_ratio': 0.1}
MAE: 20.697
Pearson correlation: 0.706
R-squared: 0.486


In [188]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 20.697
Pearson correlation: 0.706
R-squared: 0.486
Average number of non-zero coefficients: 2823.27


# Liver

In [189]:
liver_df = combined_df.loc[combined_df.iloc[:,-2] == "Liver"]

liver_df.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A5_1,100443.541866,569.498293,9197.259866,817.106246,41.267992,1034.451005,982.178215,4385.411973,935.407824,558.493495,...,101.794381,1141.747785,2.751199,511.723104,2555.864318,299.880743,448.445515,M,Liver,134
A6_1,43882.858900,853.088237,19132.800486,1147.075568,81.371493,1433.188238,1212.697740,1622.180093,1532.933939,713.969232,...,118.119910,2543.515389,307.111765,425.231675,2302.025796,782.216291,577.475114,F,Liver,52
A7_1,46308.620187,503.876178,3746.507910,867.215463,126.825977,1144.861521,1357.380725,1669.304073,970.047336,647.840801,...,41.132749,3033.540256,3.427729,414.755222,3191.215795,586.141677,661.551717,F,Liver,133
A8_1,139272.752006,589.285647,3047.097867,1149.107012,76.116063,1023.883812,1092.633804,2536.383640,989.508816,643.303498,...,130.133914,1156.473083,22.098212,564.732079,2492.187216,378.124957,414.955310,M,Liver,133
B5_1,50060.335020,509.903707,3616.007271,556.585032,32.317841,1037.761769,1052.125254,1658.982482,1138.306162,621.220713,...,25.136098,2007.296986,3.590871,373.450602,2330.475391,570.948516,416.541056,F,Liver,102


In [190]:
liver_df_male = liver_df.loc[liver_df.iloc[:,-3] == "M"]

liver_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A5_1,100443.541866,569.498293,9197.259866,817.106246,41.267992,1034.451005,982.178215,4385.411973,935.407824,558.493495,...,101.794381,1141.747785,2.751199,511.723104,2555.864318,299.880743,448.445515,M,Liver,134
A8_1,139272.752006,589.285647,3047.097867,1149.107012,76.116063,1023.883812,1092.633804,2536.383640,989.508816,643.303498,...,130.133914,1156.473083,22.098212,564.732079,2492.187216,378.124957,414.955310,M,Liver,133
B7_1,106880.160432,442.985714,13625.869465,670.276813,71.898205,1080.792369,1289.529093,2620.805531,772.325878,712.024158,...,34.789454,1338.234329,6.957891,547.354076,2435.261777,582.143529,565.908451,M,Liver,133
C5_1,113772.810228,696.735509,24082.457958,633.892699,62.842811,1158.493553,920.783791,3191.321862,857.940980,740.452247,...,166.670063,1155.761257,24.590665,478.151820,2330.648586,237.709762,437.167378,M,Liver,162
C7_1,109847.738206,626.472460,8772.952023,818.154332,91.165768,1302.034180,1491.378468,2718.609966,885.944262,883.606679,...,179.993953,766.727488,16.363087,523.618773,2966.393850,367.000658,446.478507,M,Liver,152


In [191]:
liver_sub_df_male = liver_df_male.iloc[:,:-3]
liver_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A5_1,100443.541866,569.498293,9197.259866,817.106246,41.267992,1034.451005,982.178215,4385.411973,935.407824,558.493495,...,228.349557,71.531187,723.565464,101.794381,1141.747785,2.751199,511.723104,2555.864318,299.880743,448.445515
A8_1,139272.752006,589.285647,3047.097867,1149.107012,76.116063,1023.883812,1092.633804,2536.383640,989.508816,643.303498,...,184.151765,56.473208,785.714196,130.133914,1156.473083,22.098212,564.732079,2492.187216,378.124957,414.955310
B7_1,106880.160432,442.985714,13625.869465,670.276813,71.898205,1080.792369,1289.529093,2620.805531,772.325878,712.024158,...,306.147195,104.368362,1386.939564,34.789454,1338.234329,6.957891,547.354076,2435.261777,582.143529,565.908451
C5_1,113772.810228,696.735509,24082.457958,633.892699,62.842811,1158.493553,920.783791,3191.321862,857.940980,740.452247,...,204.922209,131.150214,833.350315,166.670063,1155.761257,24.590665,478.151820,2330.648586,237.709762,437.167378
C7_1,109847.738206,626.472460,8772.952023,818.154332,91.165768,1302.034180,1491.378468,2718.609966,885.944262,883.606679,...,184.669121,39.738925,1259.957671,179.993953,766.727488,16.363087,523.618773,2966.393850,367.000658,446.478507


In [192]:
liver_sub_df_male["age"] = liver_df_male["age_days"].tolist()

liver_sub_df_male.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A5_1,100443.541866,569.498293,9197.259866,817.106246,41.267992,1034.451005,982.178215,4385.411973,935.407824,558.493495,...,71.531187,723.565464,101.794381,1141.747785,2.751199,511.723104,2555.864318,299.880743,448.445515,134
A8_1,139272.752006,589.285647,3047.097867,1149.107012,76.116063,1023.883812,1092.633804,2536.383640,989.508816,643.303498,...,56.473208,785.714196,130.133914,1156.473083,22.098212,564.732079,2492.187216,378.124957,414.955310,133
B7_1,106880.160432,442.985714,13625.869465,670.276813,71.898205,1080.792369,1289.529093,2620.805531,772.325878,712.024158,...,104.368362,1386.939564,34.789454,1338.234329,6.957891,547.354076,2435.261777,582.143529,565.908451,133
C5_1,113772.810228,696.735509,24082.457958,633.892699,62.842811,1158.493553,920.783791,3191.321862,857.940980,740.452247,...,131.150214,833.350315,166.670063,1155.761257,24.590665,478.151820,2330.648586,237.709762,437.167378,162
C7_1,109847.738206,626.472460,8772.952023,818.154332,91.165768,1302.034180,1491.378468,2718.609966,885.944262,883.606679,...,39.738925,1259.957671,179.993953,766.727488,16.363087,523.618773,2966.393850,367.000658,446.478507,152


In [193]:
liver_data_male = liver_sub_df_male.values

In [194]:
X, y = liver_data_male[:,:-1], liver_data_male[:,-1]

In [195]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 7.010e+00, tolerance: 5.022e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.846e+00, tolerance: 4.904e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 1.0, 'l1_ratio': 1.0}
Mean MAE: 17.235
Pearson correlation: 0.859
R-squared: 0.737


In [196]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 17.235
Pearson correlation: 0.859
R-squared: 0.737
Average number of non-zero coefficients: 27.45


##### Female

In [197]:
liver_df_female = liver_df.loc[liver_df.iloc[:,-3] == "F"]

liver_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,sex,tissue,age_days
A6_1,43882.858900,853.088237,19132.800486,1147.075568,81.371493,1433.188238,1212.697740,1622.180093,1532.933939,713.969232,...,118.119910,2543.515389,307.111765,425.231675,2302.025796,782.216291,577.475114,F,Liver,52
A7_1,46308.620187,503.876178,3746.507910,867.215463,126.825977,1144.861521,1357.380725,1669.304073,970.047336,647.840801,...,41.132749,3033.540256,3.427729,414.755222,3191.215795,586.141677,661.551717,F,Liver,133
B5_1,50060.335020,509.903707,3616.007271,556.585032,32.317841,1037.761769,1052.125254,1658.982482,1138.306162,621.220713,...,25.136098,2007.296986,3.590871,373.450602,2330.475391,570.948516,416.541056,F,Liver,102
B6_1,52010.405793,439.314180,3444.223173,593.074143,74.683411,1107.071734,926.952920,1555.172198,760.013532,733.654681,...,39.538276,3088.378687,4.393142,487.638740,1871.478408,505.211307,544.749583,F,Liver,134
B8_1,57870.342592,385.596632,4719.702781,704.356515,66.836750,776.334553,1033.398975,1269.898243,1043.681552,622.095900,...,87.401903,2035.950219,35.989019,493.563689,1300.745973,426.726940,483.281113,F,Liver,133


In [198]:
liver_sub_df_female = liver_df_female.iloc[:,:-3]

liver_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim6,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3
A6_1,43882.858900,853.088237,19132.800486,1147.075568,81.371493,1433.188238,1212.697740,1622.180093,1532.933939,713.969232,...,181.117195,191.616742,1007.956563,118.119910,2543.515389,307.111765,425.231675,2302.025796,782.216291,577.475114
A7_1,46308.620187,503.876178,3746.507910,867.215463,126.825977,1144.861521,1357.380725,1669.304073,970.047336,647.840801,...,140.536893,202.236017,1336.814350,41.132749,3033.540256,3.427729,414.755222,3191.215795,586.141677,661.551717
B5_1,50060.335020,509.903707,3616.007271,556.585032,32.317841,1037.761769,1052.125254,1658.982482,1138.306162,621.220713,...,118.498749,157.998332,1116.760935,25.136098,2007.296986,3.590871,373.450602,2330.475391,570.948516,416.541056
B6_1,52010.405793,439.314180,3444.223173,593.074143,74.683411,1107.071734,926.952920,1555.172198,760.013532,733.654681,...,144.973679,101.042261,856.662651,39.538276,3088.378687,4.393142,487.638740,1871.478408,505.211307,544.749583
B8_1,57870.342592,385.596632,4719.702781,704.356515,66.836750,776.334553,1033.398975,1269.898243,1043.681552,622.095900,...,107.967057,303.336017,925.431918,87.401903,2035.950219,35.989019,493.563689,1300.745973,426.726940,483.281113


In [199]:
liver_sub_df_female["age"] = liver_df_female["age_days"].tolist()

liver_sub_df_female.head(5)

,a1cf,aaas,aacs,aadat,aaed1,aagab,aak1,aamdc,aamp,aar2,...,zswim7,zswim8,zufsp,zw10,zwilch,zyg11b,zyx,zzef1,zzz3,age
A6_1,43882.858900,853.088237,19132.800486,1147.075568,81.371493,1433.188238,1212.697740,1622.180093,1532.933939,713.969232,...,191.616742,1007.956563,118.119910,2543.515389,307.111765,425.231675,2302.025796,782.216291,577.475114,52
A7_1,46308.620187,503.876178,3746.507910,867.215463,126.825977,1144.861521,1357.380725,1669.304073,970.047336,647.840801,...,202.236017,1336.814350,41.132749,3033.540256,3.427729,414.755222,3191.215795,586.141677,661.551717,133
B5_1,50060.335020,509.903707,3616.007271,556.585032,32.317841,1037.761769,1052.125254,1658.982482,1138.306162,621.220713,...,157.998332,1116.760935,25.136098,2007.296986,3.590871,373.450602,2330.475391,570.948516,416.541056,102
B6_1,52010.405793,439.314180,3444.223173,593.074143,74.683411,1107.071734,926.952920,1555.172198,760.013532,733.654681,...,101.042261,856.662651,39.538276,3088.378687,4.393142,487.638740,1871.478408,505.211307,544.749583,134
B8_1,57870.342592,385.596632,4719.702781,704.356515,66.836750,776.334553,1033.398975,1269.898243,1043.681552,622.095900,...,303.336017,925.431918,87.401903,2035.950219,35.989019,493.563689,1300.745973,426.726940,483.281113,133


In [200]:
liver_data_female = liver_sub_df_female.values

In [201]:
X, y = liver_data_female[:,:-1], liver_data_female[:,-1]

In [202]:
seed = 42

# Define the parameter grid
param_grid = {
    'alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0],
    'l1_ratio': np.arange(0, 1.1, 0.1)
}

# Define the model
model = ElasticNet(max_iter=10000, random_state=42)

# Define Leave-One-Out cross-validation
loo = LeaveOneOut()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit the GridSearchCV
grid_search.fit(X_scaled, y)

# Get the best parameters
best_params = grid_search.best_params_
print("Best parameters found: ", best_params)

# Train the model with the best parameters using Leave-One-Out cross-validation
y_true = []
y_pred = []

for train_index, test_index in loo.split(X):
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    best_model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    best_model.fit(X_train_scaled, y_train)

    y_test_pred = best_model.predict(X_test_scaled)

    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"Mean MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.434e+00, tolerance: 3.180e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(
/home/lajoyce/Documents/USQIS_2023/qiskit/lib/python3.10/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.391e+00, tolerance: 3.126e+00 Linear regression models with null weight for the l1 regularization term are more efficiently fitt

Best parameters found:  {'alpha': 0.1, 'l1_ratio': 1.0}
Mean MAE: 16.921
Pearson correlation: 0.847
R-squared: 0.702


In [203]:
# Set a seed for reproducibility
seed = 42

# Leave-One-Out cross-validation
loo = LeaveOneOut()

# Initialize lists to store actual and predicted values
y_true = []
y_pred = []
non_zero_coefs_list = []

# Loop over each train-test split
for train_index, test_index in loo.split(X):
    # Split the data
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]

    # Scale the features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Define and train the model with a random state
    model = ElasticNet(alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'], max_iter=100000, random_state=seed)
    model.fit(X_train_scaled, y_train)

    # Get the number of non-zero coefficients
    non_zero_coefs = np.sum(model.coef_ != 0)
    non_zero_coefs_list.append(non_zero_coefs)

    # Make prediction on the test set
    y_test_pred = model.predict(X_test_scaled)

    # Store the actual and predicted values
    y_true.append(y_test[0])
    y_pred.append(y_test_pred[0])

# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Calculate the Pearson correlation coefficient
pearson_corr, _ = pearsonr(y_true, y_pred)

# Calculate the R-squared value
r_squared = r2_score(y_true, y_pred)

# Calculate the mean absolute error
mae = mean_absolute_error(y_true, y_pred)

print(f"MAE: {mae:.3f}")
print(f"Pearson correlation: {pearson_corr:.3f}")
print(f"R-squared: {r_squared:.3f}")

# Print average and total non-zero coefficients across all LOO iterations
avg_non_zero_coefs = np.mean(non_zero_coefs_list)
print(f"Average number of non-zero coefficients: {avg_non_zero_coefs:.2f}")

MAE: 16.921
Pearson correlation: 0.847
R-squared: 0.702
Average number of non-zero coefficients: 24.65
